# Cross-site ESBL prediction: bidirectional external validation and transfer learning

Reproduces every number, table and figure in the revised manuscript
*Bidirectional cross-site transfer learning for prediction of extended-spectrum
beta-lactamase-producing Enterobacterales from electronic health records*
.

Pipeline
1. Cohorts from ARMD-MGB and ARMD-Stanford: CLSI-validated organisms, isolate-level
   presumptive-ESBL label (ceftriaxone / ceftazidime / aztreonam / cefpodoxime,
   Intermediate + Resistant), 46 harmonised features built with the same code at both sites.
2. Patient-level 64 / 8 / 8 / 20 split at each site (train / model-selection / calibration / held-out test).
3. Zero-shot bidirectional external validation of seven architectures
   (LR, XGBoost, FT-Transformer, MHCA-VAE, DA-VAE, FTT-DANN, TabPFN v2.6).
4. Fine-tuning data-efficiency sweep with repeated random target subsamples, XGBoost trained from
   scratch and warm-started, source-site forgetting.
5. Permutation importance on the held-out source test split; 10- and 6-feature models on identical splits.
6. Subgroups, clinical operating point, sensitivity analyses, calibration comparison, seed variability.
7. Patient-cluster bootstrap confidence intervals throughout.

Run with papermill (see README). `SMOKE=1` executes a 5 % patient subsample with reduced
epochs for an end-to-end test; `RESUME=1` reuses checkpoints under the output directory.

In [1]:
# Parameters (overridden by papermill -p).
# SMOKE: 1 = quick end-to-end test on a 5% patient subsample. RESUME: 1 = reuse checkpoints under OUT_DIR.
SMOKE = 0
RESUME = 1

In [3]:
import os, sys, gc, re, time, json, copy, math, glob, hashlib, warnings, pickle
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
warnings.filterwarnings("ignore")

import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             roc_curve, precision_recall_curve, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
import xgboost as xgb
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc

SMOKE = bool(int(SMOKE)); RESUME = bool(int(RESUME))
SEED = 42

# TabPFN licence token: read from the environment, else from a local .env (TABPFN_TOKEN or TABPFN_API_KEY).
# The token is never printed and never written anywhere by this notebook.
def _load_dotenv(path=".env"):
    if not os.path.exists(path): return
    for line in open(path):
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line: continue
        k, v = line.split("=", 1); k = k.replace("export ", "").strip(); v = v.strip().strip('"').strip("'")
        os.environ.setdefault(k, v)
_load_dotenv()
if not os.environ.get("TABPFN_TOKEN") and os.environ.get("TABPFN_API_KEY"):
    os.environ["TABPFN_TOKEN"] = os.environ["TABPFN_API_KEY"]
os.environ.setdefault("TABPFN_NO_BROWSER", "1")
os.environ.setdefault("TABPFN_DISABLE_TELEMETRY", "1")
print("TabPFN token present:", bool(os.environ.get("TABPFN_TOKEN")))
assert torch.cuda.is_available(), "A CUDA GPU is required"
DEVICE = torch.device("cuda:0")
GPU_NAME = torch.cuda.get_device_name(0)
assert "3090" in GPU_NAME, f"Expected the RTX 3090 as cuda:0, got {GPU_NAME}"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def seed_everything(seed):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    import random; random.seed(seed)

def stage_seed(*parts):
    """Deterministic seed from a description of the training call, independent of execution order."""
    h = hashlib.md5("|".join(str(p) for p in parts).encode()).hexdigest()
    return int(h[:8], 16) % (2**31 - 1)

OUT_DIR = "cross_site_outputs_smoke" if SMOKE else "cross_site_outputs"
PRED_DIR, CKPT_DIR, RES_DIR, FIG_DIR = (f"{OUT_DIR}/preds", f"{OUT_DIR}/ckpt", f"{OUT_DIR}/results", f"{OUT_DIR}/figures")
PAPER_FIG_DIR = f"{OUT_DIR}/paper_figs" if SMOKE else "figures"
for d in (PRED_DIR, CKPT_DIR, RES_DIR, FIG_DIR, PAPER_FIG_DIR): os.makedirs(d, exist_ok=True)

# ── results store (results/results.json is the single source of truth for the manuscript) ──
RESULTS_PATH = f"{RES_DIR}/results.json"
RESULTS = json.load(open(RESULTS_PATH)) if (RESUME and os.path.exists(RESULTS_PATH)) else {}

def _jsonable(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.ndarray,)): return o.tolist()
    if isinstance(o, (set,)): return sorted(o)
    if isinstance(o, (pd.Timestamp,)): return str(o)
    raise TypeError(f"not jsonable: {type(o)}")

def save_results():
    RESULTS["_meta"] = {"smoke": SMOKE, "seed": SEED, "gpu": GPU_NAME, "torch": torch.__version__,
                        "xgboost": xgb.__version__, "updated": time.strftime("%Y-%m-%d %H:%M:%S")}
    tmp = RESULTS_PATH + ".tmp"
    with open(tmp, "w") as f: json.dump(RESULTS, f, indent=1, default=_jsonable)
    os.replace(tmp, RESULTS_PATH)

def rset(*keys, value):
    """RESULTS[k1][k2]... = value (creating nested dicts)."""
    d = RESULTS
    for k in keys[:-1]: d = d.setdefault(str(k), {})
    d[str(keys[-1])] = value

def rget(*keys, default=None):
    d = RESULTS
    for k in keys:
        if not isinstance(d, dict) or str(k) not in d: return default
        d = d[str(k)]
    return d

# ── prediction and checkpoint helpers ──
def pred_path(name): return f"{PRED_DIR}/{name}.npz"
def have_preds(name): return RESUME and os.path.exists(pred_path(name))
def save_preds(name, **arrays):
    np.savez_compressed(pred_path(name), **{k: np.asarray(v, dtype=np.float32) for k, v in arrays.items()})
def load_preds(name):
    with np.load(pred_path(name)) as z: return {k: z[k] for k in z.files}
def ckpt(name): return f"{CKPT_DIR}/{name}"
def have_ckpt(name): return RESUME and os.path.exists(ckpt(name))

def save_state(name, state): torch.save({k: v.cpu() for k, v in state.items()}, ckpt(name))
def load_state(name): return torch.load(ckpt(name), map_location="cpu")

def write_tex(name, text):
    with open(f"{RES_DIR}/{name}", "w") as f: f.write(text)
    print(f"  wrote {RES_DIR}/{name}")

# ── run configuration ──
CFG = dict(
    smoke_frac=0.05 if SMOKE else 1.0,
    epochs=2 if SMOKE else 150, ft_epochs=2 if SMOKE else 60,
    patience=20, ft_patience=10,
    optuna_trials=3 if SMOKE else 30,
    n_boot=50 if SMOKE else 2000,
    budgets=[500, 1000] if SMOKE else [500, 1000, 2000, 5000],
    R_draws=2 if SMOKE else 10,          # random subsample draws per budget
    all_seeds=1 if SMOKE else 3,         # seeds for the ALL budget
    seeds=[0, 7] if SMOKE else [0, 7, 42, 123, 2024],   # zero-shot seed study
    n500_seeds=[0, 7] if SMOKE else [0, 7, 42, 123, 2024],  # init-noise study at n=500, draw 0
    tabpfn_context=2000 if SMOKE else 10000,
    perm_repeats=2 if SMOKE else 5,
)
rset("config", value=CFG)
print(f"GPU: {GPU_NAME} | free {torch.cuda.mem_get_info()[0]/1e9:.1f} GB | OUT_DIR={OUT_DIR} | SMOKE={SMOKE} RESUME={RESUME}")
print(json.dumps(CFG))

TabPFN token present: True
GPU: NVIDIA GeForce RTX 3090 | free 24.8 GB | OUT_DIR=cross_site_outputs | SMOKE=False RESUME=False
{"smoke_frac": 1.0, "epochs": 150, "ft_epochs": 60, "patience": 20, "ft_patience": 10, "optuna_trials": 30, "n_boot": 2000, "budgets": [500, 1000, 2000, 5000], "R_draws": 10, "all_seeds": 3, "seeds": [0, 7, 42, 123, 2024], "n500_seeds": [0, 7, 42, 123, 2024], "tabpfn_context": 10000, "perm_repeats": 5}


In [4]:
MGB_DIR      = "/data0/armd-mgb/physionet.org/files/armd-mgb/1.0.0"
STANFORD_DIR = "/data0/armd-stanford"
STAN_PARQUET = f"{STANFORD_DIR}/parquet"

# ── Organisms ──
ENTERO_GENERA = ["ESCHERICHIA","KLEBSIELLA","ENTEROBACTER","PROTEUS","CITROBACTER","SERRATIA",
                 "MORGANELLA","HAFNIA","PROVIDENCIA","CRONOBACTER","SALMONELLA","SHIGELLA","RAOULTELLA"]
# CLSI M100: organisms for which ESBL phenotypic screening / confirmation is validated
CLSI_ESBL_ORGANISMS = ("ESCHERICHIA COLI", "KLEBSIELLA PNEUMONIAE", "KLEBSIELLA OXYTOCA", "PROTEUS MIRABILIS")

# ── Outcome definition ──
# CLSI M100 ESBL screening agents interpretable at both sites. Cefotaxime is excluded because
# ARMD-MGB reports it with a "<= 2 ug/mL" panel floor that the CLSI-2022 re-interpretation maps to
# "Intermediate" (13,922 / 16,278 rows), which would mislabel ceftriaxone-susceptible isolates.
# Cefepime is not a CLSI screening agent. Antibiotic names are harmonised to lower case with
# non-alphanumerics collapsed to "_" (MGB `antibiotic` column, Stanford `antibiotic` column).
SCREEN_ABX = {"ceftriaxone", "ceftazidime", "aztreonam", "cefpodoxime"}
CARB_ABX   = {"meropenem", "imipenem", "ertapenem", "doripenem"}
MGB_NS_PHENO   = {"Resistant", "Intermediate", "Non-susceptible"}
MGB_TESTED     = MGB_NS_PHENO | {"Susceptible", "Susceptible dose dependent", "Susceptible dose-dependent"}
STAN_NS_SUSC   = {"Resistant", "Intermediate"}
STAN_TESTED    = STAN_NS_SUSC | {"Susceptible"}
SPECIMENS = {"cx_urine": "URINE", "cx_blood": "BLOOD", "cx_resp": "RESP"}

# ── Time windows (whole days) ──
LAG_DAYS      = 3     # prior microbiology results must be reported >= 3 days before the index order
LOOKBACK_DAYS = 730   # prior microbiology lookback
ABX_DAYS      = 90    # prior antibiotic exposure window
PROC_DAYS     = 30    # prior procedure window
MAX_GAP_DAYS  = 3650  # clip for days_since_last_cx

# ── Demographics ──
AGE_MAP = {"18-24 years":1,"25-34 years":2,"35-44 years":3,"45-54 years":4,"55-64 years":5,
           "65-74 years":6,"75-84 years":7,"85-89 years":8,"above 90":9}
AGE_65PLUS = {"65-74 years","75-84 years","85-89 years","above 90"}

# ── Antibiotic exposure classes ──
MGB_CLASS_MAP = {"T_fq_90d":["fluoroquinolone"],"T_ceph3_90d":["extended_spectrum_cephalosporin"],
    "T_carb_90d":["carbapenem"],"T_glyco_90d":["glycopeptide"],"T_sulfa_90d":["sulfonamide"],
    "T_esp_90d":["extended_spectrum_penicillin","beta_lactam_combo"],"T_amino_90d":["aminoglycoside"]}
STANFORD_SUBTYPE_MAP = {"T_fq_90d":["Fluoroquinolone"],
    "T_ceph3_90d":["Cephalosporin Gen3","Cephalosporin Gen4"],"T_carb_90d":["Carbapenem"],
    "T_glyco_90d":["Glycopeptide"],
    "T_sulfa_90d":["Sulfonamide","Sulfonamide Combo","Folate Synthesis Inhibitor"],
    "T_esp_90d":["Beta Lactam Combo"],"T_amino_90d":["Aminoglycoside"]}
PROC_MAP = [("proc_cvc","cvc"),("proc_mechvent","mechvent"),("proc_surgical","surgical_procedure")]

# ── Elixhauser comorbidity map: explicit category labels (identical vocabulary at both sites) ──
ELIX_MAP = {
    "heart_failure":        ["Congestive heart failure", "Heart failure"],
    "cardiac_arrhythmia":   ["Cardiac arrhythmias", "Cardiac dysrhythmias", "Conduction disorders"],
    "valvular_disease":     ["Valvular disease", "Nonrheumatic and unspecified valve disorders", "Chronic rheumatic heart disease"],
    "pulmonary_circulation":["Pulmonary circulation disorders", "Pulmonary heart disease"],
    "peripheral_vascular":  ["Peripheral vascular disorders", "Peripheral and visceral vascular disease"],
    "hypertension":         ["Hypertension, uncomplicated", "Hypertension, complicated", "Essential hypertension",
                             "Hypertension with complications and secondary hypertension"],
    "paralysis":            ["Paralysis", "Paralysis (other than cerebral palsy)"],
    "other_neurological":   ["Other neurological disorders"],
    "chronic_pulmonary":    ["Chronic pulmonary disease", "Chronic obstructive pulmonary disease and bronchiectasis", "Asthma"],
    "diabetes_uncomplicated":["Diabetes, uncomplicated", "Diabetes mellitus without complication"],
    "diabetes_complicated": ["Diabetes, complicated", "Diabetes mellitus with complication"],
    "hypothyroidism":       ["Hypothyroidism"],
    "renal_failure":        ["Renal failure", "Chronic kidney disease"],
    "liver_disease":        ["Liver disease", "Hepatic failure"],
    "peptic_ulcer":         ["Peptic ulcer disease excluding bleeding", "Peptic ulcer disease"],
    "aids_hiv":             ["HIV infection", "AIDS"],
    "lymphoma":             ["Lymphoma", "Hodgkin lymphoma", "Non-Hodgkin lymphoma"],
    "metastatic_cancer":    ["Metastatic cancer", "Secondary malignancies"],
    "solid_tumor":          ["Solid tumor without metastasis"],
    "rheumatoid":           ["Rheumatoid arthritis/collagen vascular diseases", "Rheumatoid arthritis and related disease",
                             "Systemic lupus erythematosus and connective tissue disorders"],
    "coagulopathy":         ["Coagulopathy", "Coagulation and hemorrhagic disorders"],
    "obesity":              ["Obesity"],
    "weight_loss":          ["Weight loss"],
    "fluid_electrolyte":    ["Fluid and electrolyte disorders"],
    "blood_loss_anemia":    ["Blood loss anemia"],
    "deficiency_anemia":    ["Deficiency anemia", "Nutritional anemia"],
    "alcohol_abuse":        ["Alcohol-related disorders", "Alcohol abuse"],
    "drug_abuse":           ["Substance-related disorders", "Drug abuse"],
    "psychoses":            ["Psychoses", "Schizophrenia spectrum and other psychotic disorders"],
    "depression":           ["Depression", "Depressive disorders"],
}
ELIX_LABEL_TO_CAT = {lab.strip().lower(): cat for cat, labs in ELIX_MAP.items() for lab in labs}
# the six binary comorbidity features used by the models
COMORBID_FEATURES = {"comorbid_heart_failure":"heart_failure", "comorbid_liver_disease":"liver_disease",
                     "comorbid_lymphoma":"lymphoma", "comorbid_metastatic_cancer":"metastatic_cancer",
                     "comorbid_obesity":"obesity", "comorbid_renal_failure":"renal_failure"}

# ── Feature set (46 harmonised features; org_entero / org_other dropped: constant 0 in the CLSI cohort) ──
FEATURE_DICT = {
    "demographics": ["age_encoded","gender_male","age_65plus","adi_score_clean","adi_missing","adi_high"],
    "comorbidities": ["comorbid_heart_failure","comorbid_liver_disease","comorbid_lymphoma",
        "comorbid_metastatic_cancer","comorbid_obesity","comorbid_renal_failure","elixhauser_count"],
    "ward": ["hosp_ward_IP","hosp_ward_OP","hosp_ward_ER"],
    "antibiotics": ["T_fq_90d","T_ceph3_90d","T_carb_90d","T_glyco_90d","T_sulfa_90d","T_esp_90d","T_amino_90d"],
    "resistance": ["prior_ESBL","num_prior_orgs","days_since_prior_org","prior_esbl_ast","prior_carb_resist","prior_n_resistant_abx"],
    "procedures": ["proc_cvc","proc_mechvent","proc_surgical","any_proc_30d"],
    "culture_organism": ["cx_urine","cx_blood","cx_resp","org_ecoli","org_klebsiella","org_proteus"],
    "temporal_interactions": ["days_since_last_cx","has_prior_cx","fq_and_ceph3","any_immunocomp","any_invasive_proc","prior_esbl_and_ceph3","hospital_multiabx"],
}
FEATURES = [f for g in FEATURE_DICT.values() for f in g]
N_FEAT = len(FEATURES)
GROUP_SIZES = [len(g) for g in FEATURE_DICT.values()]
GROUP_INDICES, _c = [], 0
for _s in GROUP_SIZES:
    GROUP_INDICES.append(list(range(_c, _c+_s))); _c += _s
FEAT_TO_GROUP = {f: g for g, fs in FEATURE_DICT.items() for f in fs}
assert N_FEAT == 46, N_FEAT
ID_COLS = {"order_id","patient_id","order_dt","culture_description","organism","site","ESBL","split",
           "n_screen_tested","polymicrobial","n_clsi_isolates","esbl_anyorg","esbl_confirm","age_group",
           "adi_score","prior_cx_30d","first_culture"}
print(f"Features: {N_FEAT} in {len(GROUP_SIZES)} groups {GROUP_SIZES}; screening agents {sorted(SCREEN_ABX)}; lag {LAG_DAYS} d, lookback {LOOKBACK_DAYS} d")
rset("definitions", value={"screening_agents": sorted(SCREEN_ABX), "carbapenems": sorted(CARB_ABX),
     "clsi_organisms": list(CLSI_ESBL_ORGANISMS), "lag_days": LAG_DAYS, "lookback_days": LOOKBACK_DAYS,
     "abx_days": ABX_DAYS, "proc_days": PROC_DAYS, "n_features": N_FEAT, "features": FEATURES,
     "elixhauser_categories": list(ELIX_MAP)})

Features: 46 in 8 groups [6, 7, 3, 7, 6, 4, 6, 7]; screening agents ['aztreonam', 'cefpodoxime', 'ceftazidime', 'ceftriaxone']; lag 3 d, lookback 730 d


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# Shared, site-agnostic builders: labels, prior-microbiology features, comorbidities,
# patient-level splits, Table 1. Both sites call exactly these functions.
# ══════════════════════════════════════════════════════════════════════════════
def norm_abx(s):
    """Harmonise antibiotic names: lower case, non-alphanumerics -> '_' (e.g. 'piperacillin/tazobactam' and
    'piperacillin_tazobactam' both become 'piperacillin_tazobactam')."""
    return (s.astype(str).str.lower().str.strip()
             .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))

def canon_org(s):
    """Upper-case organism name without parenthetical qualifiers, e.g.
    'Klebsiella pneumoniae (carbapenem resistant)' -> 'KLEBSIELLA PNEUMONIAE'."""
    s = s.astype(str).str.upper().str.strip()
    return s.str.replace(r"\s*\([^)]*\)", "", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()

def genus_of(s): return s.str.split(" ").str[0]
def is_clsi_org(s): return s.str.startswith(CLSI_ESBL_ORGANISMS)
def is_entero_org(s): return genus_of(s).isin(ENTERO_GENERA)

def build_labels(ast):
    """Isolate-level presumptive-ESBL label per culture order.

    ast: one row per (order, organism, antibiotic) with columns
         order_id, org (canonical), abx (harmonised), ns (non-susceptible), tested, is_clsi, is_entero.
    Returns one row per order that has >= 1 CLSI-validated isolate."""
    clsi = ast[ast["is_clsi"]]
    scr = clsi[clsi["abx"].isin(SCREEN_ABX)]
    idx = pd.Index(clsi["order_id"].unique(), name="order_id")
    g = pd.DataFrame(index=idx)
    g["ESBL"] = scr.groupby("order_id")["ns"].any().reindex(idx).fillna(False).astype(int)
    g["n_screen_tested"] = scr[scr["tested"]].groupby("order_id")["abx"].nunique().reindex(idx).fillna(0).astype(int)
    g["n_clsi_isolates"] = clsi.groupby("order_id")["org"].nunique().reindex(idx).fillna(0).astype(int)
    g["polymicrobial"] = (ast.groupby("order_id")["org"].nunique().reindex(idx) > 1).astype(int)
    for feat, prefix in [("org_ecoli", "ESCHERICHIA COLI"), ("org_klebsiella", "KLEBSIELLA"), ("org_proteus", "PROTEUS MIRABILIS")]:
        g[feat] = clsi[clsi["org"].str.startswith(prefix)].groupby("order_id").size().reindex(idx).notna().astype(int)
    g["organism"] = clsi.groupby("order_id")["org"].min()
    # diagnostic: label under the previous any-organism-in-order rule
    scr_any = ast[ast["abx"].isin(SCREEN_ABX)]
    g["esbl_anyorg"] = scr_any.groupby("order_id")["ns"].any().reindex(idx).fillna(False).astype(int)
    return g.reset_index()

def build_prior_features(index_df, ast, orders, orgs):
    """Prior-microbiology features from a patient-level self-join of the site's culture table.

    index_df: order_id, patient_id, order_dt (day-normalised Timestamp) for the cohort cultures
    ast:      all AST rows at the site (any specimen, any organism)  [order_id, org, abx, ns, is_clsi, is_entero]
    orders:   all culture orders at the site [order_id, patient_id, order_dt, positive]
    orgs:     all identified organisms [order_id, org]
    Prior results count only if reported >= LAG_DAYS and <= LOOKBACK_DAYS before the index order."""
    scr = ast[ast["abx"].isin(SCREEN_ABX) & ast["ns"]]
    flags = pd.DataFrame({"order_id": orders["order_id"]})
    flags["esbl_clsi"] = flags["order_id"].isin(set(scr.loc[scr["is_clsi"], "order_id"]))
    flags["esbl_ent"]  = flags["order_id"].isin(set(scr.loc[scr["is_entero"], "order_id"]))
    flags["carb_ns"]   = flags["order_id"].isin(set(ast.loc[ast["abx"].isin(CARB_ABX) & ast["ns"], "order_id"]))
    o = orders[["order_id", "patient_id", "order_dt", "positive"]].merge(flags, on="order_id")
    o = o.rename(columns={"order_id": "prior_id", "order_dt": "prior_dt"})
    idx = index_df[["order_id", "patient_id", "order_dt"]].rename(columns={"order_id": "idx_id", "order_dt": "idx_dt"})
    pairs = idx.merge(o, on="patient_id", how="inner")
    pairs = pairs[pairs["prior_id"] != pairs["idx_id"]]
    pairs["gap"] = (pairs["idx_dt"] - pairs["prior_dt"]).dt.days
    out = pd.DataFrame(index=pd.Index(idx["idx_id"].values, name="order_id"))

    # any earlier culture order (the existence of a culture is known at order time)
    prev_gap = pairs.loc[pairs["gap"] > 0].groupby("idx_id")["gap"].min().reindex(out.index)
    out["days_since_last_cx"] = prev_gap.fillna(0).clip(0, MAX_GAP_DAYS)
    out["has_prior_cx"] = (out["days_since_last_cx"] > 0).astype(int)
    out["prior_cx_30d"] = (prev_gap <= 30).fillna(False).astype(int)   # for the episode sensitivity analysis

    # prior microbiology results available at prediction time
    win = pairs[(pairs["gap"] >= LAG_DAYS) & (pairs["gap"] <= LOOKBACK_DAYS)]
    agg = win.groupby("idx_id").agg(prior_ESBL=("esbl_clsi", "any"), prior_esbl_ast=("esbl_ent", "any"),
                                    prior_carb_resist=("carb_ns", "any")).reindex(out.index)
    for c in agg.columns: out[c] = agg[c].fillna(False).astype(int)
    out["days_since_prior_org"] = win[win["positive"]].groupby("idx_id")["gap"].min().reindex(out.index).fillna(0)
    ns_pairs = ast.loc[ast["ns"], ["order_id", "abx"]].drop_duplicates().rename(columns={"order_id": "prior_id"})
    m = win[["idx_id", "prior_id"]].merge(ns_pairs, on="prior_id")
    out["prior_n_resistant_abx"] = m.groupby("idx_id")["abx"].nunique().reindex(out.index).fillna(0).astype(int)
    gen = orgs[["order_id", "org"]].drop_duplicates().copy(); gen["genus"] = genus_of(gen["org"])
    gen = gen[["order_id", "genus"]].drop_duplicates().rename(columns={"order_id": "prior_id"})
    m2 = win[["idx_id", "prior_id"]].merge(gen, on="prior_id")
    out["num_prior_orgs"] = m2.groupby("idx_id")["genus"].nunique().reindex(out.index).fillna(0).astype(int)
    return out.reset_index()

def comorbidity_features(cmb, ids, site):
    """cmb: [order_id, category] rows (diagnoses recorded before the culture where the source provides timing).
    Maps category labels to Elixhauser categories with the explicit ELIX_MAP; unmatched labels are ignored."""
    cmb = cmb[cmb["order_id"].isin(ids)].copy()
    cmb["elix"] = cmb["category"].astype(str).str.strip().str.lower().map(ELIX_LABEL_TO_CAT)
    matched = cmb.dropna(subset=["elix"])
    print(f"  {site} comorbidity: {len(cmb):,} rows, {cmb['category'].nunique()} distinct labels; "
          f"{matched['category'].nunique()} labels / {len(matched):,} rows map to {matched['elix'].nunique()} Elixhauser categories")
    rset("cohort", site, "elix_matched_labels", value=sorted(matched["category"].astype(str).unique()))
    rset("cohort", site, "elix_unmatched_top", value=cmb.loc[cmb["elix"].isna(), "category"].value_counts().head(20).to_dict())
    per = matched.groupby("order_id")["elix"].agg(lambda s: set(s))
    out = pd.DataFrame(index=pd.Index(list(ids), name="order_id"))
    for feat, cat in COMORBID_FEATURES.items():
        out[feat] = per.apply(lambda s: cat in s).reindex(out.index).fillna(False).astype(int)
    out["elixhauser_count"] = per.apply(len).reindex(out.index).fillna(0).astype(int)
    return out.reset_index()

def load_stan_comorbidity(sids):
    """Stanford comorbidity components from the parquet mirror (order ids are strings there).
    start_days >= 0 for every row in this table, i.e. components start on or before the culture day."""
    ids = pa.array([str(int(i)) for i in sids])
    parts = []
    files = sorted(glob.glob(f"{STAN_PARQUET}/comorbidity/*.parquet"))
    assert files, "Stanford comorbidity parquet mirror not found"
    for f in files:
        pf = pq.ParquetFile(f)
        for batch in pf.iter_batches(batch_size=2_000_000, columns=["order_proc_id_coded", "comorbidity_component",
                                                                     "comorbidity_component_start_days_culture"]):
            sub = batch.filter(pc.is_in(batch.column("order_proc_id_coded"), value_set=ids))
            if sub.num_rows: parts.append(sub.to_pandas())
    cmb = pd.concat(parts, ignore_index=True)
    cmb["order_id"] = cmb["order_proc_id_coded"].astype(np.int64)
    start = pd.to_numeric(cmb["comorbidity_component_start_days_culture"], errors="coerce")
    cmb = cmb[start.fillna(0) >= 0]
    return cmb.rename(columns={"comorbidity_component": "category"})[["order_id", "category"]]

def make_site_splits(df, seed=SEED, fractions=(0.64, 0.08, 0.08, 0.20), names=("train", "val_sel", "val_cal", "test")):
    """Patient-level split; depends only on the sorted set of patient ids and the seed."""
    pids = np.array(sorted(df["patient_id"].astype(str).unique()))
    rng = np.random.RandomState(seed); rng.shuffle(pids)
    cuts = np.round(np.cumsum(fractions) * len(pids)).astype(int)
    assign = {}
    start = 0
    for name, end in zip(names, cuts):
        assign.update({p: name for p in pids[start:end]}); start = end
    return df["patient_id"].astype(str).map(assign)

def specimen_flags(df):
    cx = df["culture_description"].astype(str).str.upper()
    for feat, key in SPECIMENS.items(): df[feat] = cx.str.contains(key, na=False).astype(int)
    return df

def derived_features(df):
    df["fq_and_ceph3"] = ((df["T_fq_90d"] > 0) & (df["T_ceph3_90d"] > 0)).astype(int)
    df["any_immunocomp"] = ((df["comorbid_lymphoma"] > 0) | (df["comorbid_metastatic_cancer"] > 0)).astype(int)
    df["any_invasive_proc"] = ((df["proc_cvc"] > 0) | (df["proc_mechvent"] > 0)).astype(int)
    df["prior_esbl_and_ceph3"] = ((df["prior_ESBL"] > 0) & (df["T_ceph3_90d"] > 0)).astype(int)
    df["hospital_multiabx"] = ((df["hosp_ward_IP"] > 0) &
        (df[["T_fq_90d","T_ceph3_90d","T_carb_90d","T_glyco_90d","T_sulfa_90d","T_esp_90d","T_amino_90d"]].sum(axis=1) >= 2)).astype(int)
    df["any_proc_30d"] = ((df["proc_cvc"] + df["proc_mechvent"] + df["proc_surgical"]) > 0).astype(int)
    return df

def finalize_site(df, site):
    for f in FEATURES:
        if f not in df.columns: df[f] = 0
        df[f] = pd.to_numeric(df[f], errors="coerce").fillna(0)
    df["site"] = site
    assert df["order_id"].duplicated().sum() == 0
    return df.sort_values(["patient_id", "order_dt", "order_id"]).reset_index(drop=True)

def cohort_diagnostics(df, site, t0):
    d = {"n_cultures": len(df), "n_patients": df["patient_id"].nunique(), "n_esbl": int(df["ESBL"].sum()),
         "prevalence": float(df["ESBL"].mean()),
         "n_screen_tested_ge1": int((df["n_screen_tested"] >= 1).sum()),
         "n_multi_clsi_isolate": int((df["n_clsi_isolates"] > 1).sum()),
         "n_polymicrobial": int(df["polymicrobial"].sum()),
         "n_label_only_via_non_clsi_isolate": int(((df["esbl_anyorg"] == 1) & (df["ESBL"] == 0)).sum()),
         "prior_esbl_x_prior_esbl_ast": pd.crosstab(df["prior_ESBL"], df["prior_esbl_ast"]).to_dict(),
         "cultures_per_patient_mean": float(df.groupby("patient_id").size().mean())}
    if "esbl_confirm" in df.columns and df["esbl_confirm"].notna().any():
        c = df[df["esbl_confirm"].notna()]
        ct = pd.crosstab(c["ESBL"], c["esbl_confirm"].astype(int))
        d["confirmatory"] = {"n_with_result": int(len(c)), "n_confirm_pos": int(c["esbl_confirm"].sum()),
                             "crosstab_presumptive_x_confirm": ct.to_dict(),
                             "ppv_presumptive": float(((c["ESBL"] == 1) & (c["esbl_confirm"] == 1)).sum() / max((c["ESBL"] == 1).sum(), 1)),
                             "sens_presumptive": float(((c["ESBL"] == 1) & (c["esbl_confirm"] == 1)).sum() / max((c["esbl_confirm"] == 1).sum(), 1)),
                             "spec_presumptive": float(((c["ESBL"] == 0) & (c["esbl_confirm"] == 0)).sum() / max((c["esbl_confirm"] == 0).sum(), 1))}
    for k, v in d.items():
        if not isinstance(v, dict): rset("cohort", site, k, value=v)
        else: rset("cohort", site, k, value=v)
    print(f"  {site}: {d['n_cultures']:,} cultures, {d['n_patients']:,} patients | presumptive ESBL+ {d['n_esbl']:,} ({d['prevalence']:.2%}) | "
          f">=1 screening agent tested {d['n_screen_tested_ge1']:,} | multi-CLSI-isolate {d['n_multi_clsi_isolate']:,} | "
          f"label negative only because the non-susceptible isolate was not a CLSI organism: {d['n_label_only_via_non_clsi_isolate']:,} | {time.time()-t0:.0f}s")
    if "confirmatory" in d:
        cf = d["confirmatory"]
        print(f"  {site} confirmatory ESBL test available for {cf['n_with_result']:,} cultures ({cf['n_confirm_pos']:,} positive): "
              f"presumptive PPV {cf['ppv_presumptive']:.3f}, sensitivity {cf['sens_presumptive']:.3f}, specificity {cf['spec_presumptive']:.3f}")

## MGB cohort and features (ARMD-MGB, PhysioNet)

Order times in ARMD-MGB are date-shifted per patient, so only within-patient intervals are meaningful.
The comorbidity table carries no diagnosis timing (documented as a limitation); Elixhauser categories are
mapped with the same explicit label list used at Stanford.

In [6]:
t0 = time.time(); print("MGB feature engineering")
micro = pd.read_csv(f"{MGB_DIR}/microbiology_cohort_deid_tj_updated.csv", low_memory=False,
    usecols=["anon_id","order_proc_id_coded","culture_description","organism","has_AST","AST_code","antibiotic",
             "CLSI_2022_pheno","AST_pheno","AST_val1","enzyme_class","order_time_jittered_utc_shifted"])
micro = micro.rename(columns={"anon_id": "patient_id", "order_proc_id_coded": "order_id"})
micro["patient_id"] = micro["patient_id"].astype(str)
micro["order_dt"] = pd.to_datetime(micro["order_time_jittered_utc_shifted"], errors="coerce").dt.normalize()
micro["has_org"] = micro["organism"].notna()

# all culture orders at the site (positive or negative), all identified organisms, all AST rows
orders_mgb = (micro.groupby("order_id").agg(patient_id=("patient_id", "first"), order_dt=("order_dt", "first"),
              culture_description=("culture_description", "first"), positive=("has_org", "max")).reset_index())
orgs_mgb = micro.loc[micro["has_org"], ["order_id", "organism"]].copy()
orgs_mgb["org"] = canon_org(orgs_mgb["organism"]); orgs_mgb = orgs_mgb[["order_id", "org"]].drop_duplicates()
ast_mgb = micro[(micro["has_AST"] == "X") & micro["has_org"] & micro["antibiotic"].notna()].copy()
ast_mgb["org"] = canon_org(ast_mgb["organism"]); ast_mgb["abx"] = norm_abx(ast_mgb["antibiotic"])
# Non-susceptibility uses the CLSI-2022 re-interpretation supplied with ARMD-MGB where an MIC/disk value exists.
# Where CLSI_2022_pheno is blank (the laboratory issued a category with no numeric value), the laboratory-reported
# category (AST_pheno) is used instead: every breakpoint revision for these agents lowered the thresholds, so a
# result the laboratory called Intermediate/Resistant is also non-susceptible under the 2022 criteria.
_clsi = ast_mgb["CLSI_2022_pheno"]; _lab = ast_mgb["AST_pheno"]
ast_mgb["ns_clsi_only"] = _clsi.isin(MGB_NS_PHENO)
ast_mgb["ns"] = ast_mgb["ns_clsi_only"] | (_clsi.isna() & _lab.isin(MGB_NS_PHENO))
ast_mgb["tested"] = _clsi.isin(MGB_TESTED) | (_clsi.isna() & _lab.isin(MGB_TESTED))
ast_mgb["is_clsi"] = is_clsi_org(ast_mgb["org"]); ast_mgb["is_entero"] = is_entero_org(ast_mgb["org"])
_fb = ast_mgb["is_clsi"] & ast_mgb["abx"].isin(SCREEN_ABX) & _clsi.isna() & _lab.isin(MGB_NS_PHENO)
rset("cohort", "MGB", "n_screen_rows_lab_phenotype_fallback", value=int(_fb.sum()))
# agreement of the two MGB categories on validated-isolate screening-agent results (reported in Methods)
_scr = ast_mgb["is_clsi"] & ast_mgb["abx"].isin(SCREEN_ABX)
_both = _scr & _clsi.isin(MGB_TESTED) & _lab.isin(MGB_TESTED)
_dis = _both & (_clsi.isin(MGB_NS_PHENO) != _lab.isin(MGB_NS_PHENO))
_val = pd.to_numeric(ast_mgb["AST_val1"], errors="coerce")
rset("cohort", "MGB", "screen_agent_categories", value={
    "n_rows": int(_scr.sum()), "n_no_value": int((_scr & _clsi.isna()).sum()),
    "n_both_categories": int(_both.sum()), "pct_agree_s_vs_ns": float(1 - _dis.sum() / max(_both.sum(), 1)),
    "n_discordant": int(_dis.sum()), "n_discordant_lab_s_clsi_ns_at_8": int((_dis & ~_lab.isin(MGB_NS_PHENO) & (_val == 8)).sum()),
    "n_cefotaxime_rows": int((ast_mgb["is_clsi"] & (ast_mgb["abx"] == "cefotaxime")).sum()),
    "n_cefotaxime_intermediate": int((ast_mgb["is_clsi"] & (ast_mgb["abx"] == "cefotaxime") & (_clsi == "Intermediate")).sum())})
_strict = ast_mgb[["order_id", "org", "abx", "tested", "is_clsi", "is_entero"]].copy(); _strict["ns"] = ast_mgb["ns_clsi_only"]
ast_mgb = ast_mgb[["order_id", "org", "abx", "ns", "tested", "is_clsi", "is_entero"]].reset_index(drop=True)
# confirmatory ESBL test result (enzyme_class == "ESBL"; AST_pheno Positive / Negative)
conf = micro[micro["enzyme_class"] == "ESBL"].groupby("order_id")["AST_pheno"].agg(
    lambda s: 1.0 if (s == "Positive").any() else (0.0 if (s == "Negative").any() else np.nan))
del micro; gc.collect()

labels = build_labels(ast_mgb)
_ls = build_labels(_strict).set_index("order_id")["ESBL"]
_nflip = int(((labels["ESBL"] == 1) & (labels["order_id"].map(_ls) == 0)).sum())
rset("cohort", "MGB", "n_positive_only_via_lab_phenotype", value=_nflip)
print(f"  MGB: {_nflip:,} cultures are presumptive-ESBL-positive only through a laboratory-reported category with no CLSI-2022 value")
del _strict, _ls
mgb = labels.merge(orders_mgb[["order_id", "patient_id", "order_dt", "culture_description"]], on="order_id", how="left")
n0 = len(mgb)
mgb = specimen_flags(mgb)
mgb = mgb[(mgb["cx_urine"] + mgb["cx_blood"] + mgb["cx_resp"]) > 0]
n1 = len(mgb)

# Demographics (adults only)
demo = pd.read_csv(f"{MGB_DIR}/demographics_deid_tj.csv", low_memory=False).drop_duplicates("order_proc_id_coded").rename(columns={"order_proc_id_coded": "order_id"})
demo["age_encoded"] = demo["age"].map(AGE_MAP)
demo["age_65plus"] = demo["age"].isin(AGE_65PLUS).astype(int)
demo["gender_male"] = (demo["gender"].astype(str).str.strip().str.lower() == "male").astype(int)
mgb = mgb.merge(demo[["order_id", "age", "age_encoded", "age_65plus", "gender_male"]], on="order_id", how="left")
mgb = mgb[mgb["age_encoded"].notna()].copy(); mgb["age_encoded"] = mgb["age_encoded"].astype(int)
mgb = mgb.rename(columns={"age": "age_group"})
n2 = len(mgb)
print(f"  CLSI-organism cultures {n0:,} -> urine/blood/respiratory {n1:,} -> adults with known age {n2:,}")
rset("cohort", "MGB", "n_clsi_orders", value=n0); rset("cohort", "MGB", "n_after_specimen", value=n1); rset("cohort", "MGB", "n_after_adult", value=n2)
cids = set(mgb["order_id"])

# ADI (threshold = 80th percentile of the MGB ADI distribution, applied to both sites)
adi = pd.read_csv(f"{MGB_DIR}/ADI_deid_tj.csv", low_memory=False).drop_duplicates("order_proc_id_coded").rename(columns={"order_proc_id_coded": "order_id"})
adi["adi_score"] = pd.to_numeric(adi["adi_score"], errors="coerce")
ADI_HIGH_THRESHOLD = float(adi["adi_score"].quantile(0.80))
adi["adi_score_clean"] = adi["adi_score"].fillna(0); adi["adi_missing"] = adi["adi_score"].isna().astype(int)
adi["adi_high"] = (adi["adi_score"] > ADI_HIGH_THRESHOLD).fillna(False).astype(int)
mgb = mgb.merge(adi[["order_id", "adi_score", "adi_score_clean", "adi_missing", "adi_high"]], on="order_id", how="left")
mgb["adi_missing"] = mgb["adi_missing"].fillna(1).astype(int)
rset("definitions", "adi_high_threshold", value=ADI_HIGH_THRESHOLD)

# Comorbidities (no diagnosis timing available in ARMD-MGB)
cmb = pd.read_csv(f"{MGB_DIR}/comorbidity_deid_tj.csv", low_memory=False, usecols=["order_proc_id_coded", "category"]).rename(columns={"order_proc_id_coded": "order_id"})
mgb = mgb.merge(comorbidity_features(cmb, cids, "MGB"), on="order_id", how="left"); del cmb; gc.collect()

# Ward
ward = pd.read_csv(f"{MGB_DIR}/ward_type_deid_tj.csv", low_memory=False).drop_duplicates("order_proc_id_coded").rename(columns={"order_proc_id_coded": "order_id"})
mgb = mgb.merge(ward[["order_id", "hosp_ward_IP", "hosp_ward_OP", "hosp_ward_ER"]], on="order_id", how="left")

# Prior antibiotics (90 d)
abx = pd.read_csv(f"{MGB_DIR}/prior_abx_deid_tj.csv", low_memory=False, usecols=["order_proc_id_coded", "last_dose_to_culture", "drug_class"]).rename(columns={"order_proc_id_coded": "order_id"})
abx = abx[abx["order_id"].isin(cids)]
abx["days"] = pd.to_numeric(abx["last_dose_to_culture"], errors="coerce")
abx = abx[(abx["days"] > 0) & (abx["days"] <= ABX_DAYS)]
dc = abx["drug_class"].astype(str).str.lower().str.strip()
for feat, classes in MGB_CLASS_MAP.items():
    mgb[feat] = mgb["order_id"].isin(set(abx.loc[dc.isin(classes), "order_id"])).astype(int)
del abx; gc.collect()

# Prior procedures (30 d)
proc = pd.read_csv(f"{MGB_DIR}/prior_procedures_deid_tj.csv", low_memory=False).rename(columns={"order_proc_id_coded": "order_id"})
proc = proc[proc["order_id"].isin(cids)]
proc["days"] = pd.to_numeric(proc["procedure_days_culture"], errors="coerce")
proc = proc[(proc["days"] > 0) & (proc["days"] <= PROC_DAYS)]
pdesc = proc["procedure_description"].astype(str).str.strip().str.lower()
for feat, key in PROC_MAP:
    mgb[feat] = mgb["order_id"].isin(set(proc.loc[pdesc == key, "order_id"])).astype(int)
del proc; gc.collect()

# Prior microbiology (harmonised self-join)
mgb = mgb.merge(build_prior_features(mgb, ast_mgb, orders_mgb, orgs_mgb), on="order_id", how="left")
mgb["esbl_confirm"] = mgb["order_id"].map(conf)
mgb = derived_features(mgb)
mgb = finalize_site(mgb, "MGB")
cohort_diagnostics(mgb, "MGB", t0)

MGB feature engineering


  MGB: 1,969 cultures are presumptive-ESBL-positive only through a laboratory-reported category with no CLSI-2022 value


  CLSI-organism cultures 120,742 -> urine/blood/respiratory 120,742 -> adults with known age 120,742


  MGB comorbidity: 737,358 rows, 484 distinct labels; 53 labels / 291,633 rows map to 28 Elixhauser categories


  MGB: 120,742 cultures, 67,185 patients | presumptive ESBL+ 16,677 (13.81%) | >=1 screening agent tested 102,742 | multi-CLSI-isolate 3,702 | label negative only because the non-susceptible isolate was not a CLSI organism: 885 | 53s
  MGB confirmatory ESBL test available for 8,166 cultures (558 positive): presumptive PPV 0.887, sensitivity 0.975, specificity 0.991


## Stanford cohort and features (ARMD-Stanford, Dryad)

Comorbidity components are read from the parquet mirror of `microbiology_cultures_comorbidity.csv`
(components start on or before the culture day by construction). Prior-microbiology features use the same
self-join code as MGB.

In [7]:
t0 = time.time(); print("Stanford feature engineering")
cul = pd.read_csv(f"{STANFORD_DIR}/microbiology_cultures_cohort.csv", low_memory=False,
    usecols=["anon_id","order_proc_id_coded","order_time_jittered_utc","culture_description","was_positive","organism","antibiotic","susceptibility"])
cul = cul.rename(columns={"anon_id": "patient_id", "order_proc_id_coded": "order_id"})
cul["patient_id"] = cul["patient_id"].astype(str)
cul["order_dt"] = pd.to_datetime(cul["order_time_jittered_utc"], utc=True, errors="coerce").dt.tz_convert(None).dt.normalize()
# ARMD-Stanford stores the literal string "Null" (not a missing value) in `organism` for negative cultures;
# it must not be treated as an organism (it would inflate num_prior_orgs and make every order "positive").
cul["has_org"] = cul["organism"].notna() & (cul["organism"].astype(str).str.strip().str.lower() != "null")
cul["pos"] = cul["has_org"] | (pd.to_numeric(cul["was_positive"], errors="coerce") == 1)

orders_stan = (cul.groupby("order_id").agg(patient_id=("patient_id", "first"), order_dt=("order_dt", "first"),
               culture_description=("culture_description", "first"), positive=("pos", "max")).reset_index())
orgs_stan = cul.loc[cul["has_org"], ["order_id", "organism"]].copy()
orgs_stan["org"] = canon_org(orgs_stan["organism"]); orgs_stan = orgs_stan[["order_id", "org"]].drop_duplicates()
susc = cul["susceptibility"].astype(str).str.strip()
ast_stan = cul[cul["has_org"] & cul["antibiotic"].notna() & susc.isin(STAN_TESTED)].copy()
ast_stan["org"] = canon_org(ast_stan["organism"]); ast_stan["abx"] = norm_abx(ast_stan["antibiotic"])
ast_stan["ns"] = ast_stan["susceptibility"].astype(str).str.strip().isin(STAN_NS_SUSC); ast_stan["tested"] = True
ast_stan["is_clsi"] = is_clsi_org(ast_stan["org"]); ast_stan["is_entero"] = is_entero_org(ast_stan["org"])
ast_stan = ast_stan[["order_id", "org", "abx", "ns", "tested", "is_clsi", "is_entero"]].reset_index(drop=True)
del cul; gc.collect()

labels = build_labels(ast_stan)
stan = labels.merge(orders_stan[["order_id", "patient_id", "order_dt", "culture_description"]], on="order_id", how="left")
n0 = len(stan)
stan = specimen_flags(stan)
stan = stan[(stan["cx_urine"] + stan["cx_blood"] + stan["cx_resp"]) > 0]
n1 = len(stan)

# Demographics (adults only; gender coded 1 = male in ARMD-Stanford)
dem = pd.read_csv(f"{STANFORD_DIR}/microbiology_cultures_demographics.csv", low_memory=False).drop_duplicates("order_proc_id_coded").rename(columns={"order_proc_id_coded": "order_id"})
dem["age_encoded"] = dem["age"].map(AGE_MAP)
dem["age_65plus"] = dem["age"].isin(AGE_65PLUS).astype(int)
dem["gender_male"] = (pd.to_numeric(dem["gender"].replace("Null", np.nan), errors="coerce") == 1).astype(int)
stan = stan.merge(dem[["order_id", "age", "age_encoded", "age_65plus", "gender_male"]], on="order_id", how="left")
stan = stan[stan["age_encoded"].notna()].copy(); stan["age_encoded"] = stan["age_encoded"].astype(int)
stan = stan.rename(columns={"age": "age_group"})
n2 = len(stan)
gmr = stan["gender_male"].mean(); assert 0.1 < gmr < 0.9, f"gender_male={gmr:.2%}: likely mismapped"
print(f"  CLSI-organism cultures {n0:,} -> urine/blood/respiratory {n1:,} -> adults with known age {n2:,}")
rset("cohort", "Stanford", "n_clsi_orders", value=n0); rset("cohort", "Stanford", "n_after_specimen", value=n1); rset("cohort", "Stanford", "n_after_adult", value=n2)
sids = set(stan["order_id"])

# ADI (MGB-derived threshold)
adi = pd.read_csv(f"{STANFORD_DIR}/microbiology_cultures_adi_scores.csv", low_memory=False).drop_duplicates("order_proc_id_coded").rename(columns={"order_proc_id_coded": "order_id"})
adi["adi_score"] = pd.to_numeric(adi["adi_score"], errors="coerce")
adi["adi_score_clean"] = adi["adi_score"].fillna(0); adi["adi_missing"] = adi["adi_score"].isna().astype(int)
adi["adi_high"] = (adi["adi_score"] > ADI_HIGH_THRESHOLD).fillna(False).astype(int)
stan = stan.merge(adi[["order_id", "adi_score", "adi_score_clean", "adi_missing", "adi_high"]], on="order_id", how="left")
stan["adi_missing"] = stan["adi_missing"].fillna(1).astype(int)

# Ward
ward = pd.read_csv(f"{STANFORD_DIR}/microbiology_cultures_ward_info.csv", low_memory=False).drop_duplicates("order_proc_id_coded").rename(columns={"order_proc_id_coded": "order_id"})
stan = stan.merge(ward[["order_id", "hosp_ward_IP", "hosp_ward_OP", "hosp_ward_ER"]], on="order_id", how="left")

# Comorbidities (parquet mirror, components starting on/before the culture day)
stan = stan.merge(comorbidity_features(load_stan_comorbidity(sids), sids, "Stanford"), on="order_id", how="left")

# Prior antibiotics (90 d)
abx = pd.read_csv(f"{STANFORD_DIR}/microbiology_cultures_antibiotic_subtype_exposure.csv", low_memory=False,
                  usecols=["order_proc_id_coded", "antibiotic_subtype", "medication_time_to_cultureTime"]).rename(columns={"order_proc_id_coded": "order_id"})
abx = abx[abx["order_id"].isin(sids)]
abx["days"] = pd.to_numeric(abx["medication_time_to_cultureTime"].replace("Null", np.nan), errors="coerce")
abx = abx[(abx["days"] > 0) & (abx["days"] <= ABX_DAYS)]
subtype = abx["antibiotic_subtype"].astype(str).str.strip()
for feat, subtypes in STANFORD_SUBTYPE_MAP.items():
    stan[feat] = stan["order_id"].isin(set(abx.loc[subtype.isin(subtypes), "order_id"])).astype(int)
del abx; gc.collect()

# Prior procedures (30 d)
proc = pd.read_csv(f"{STANFORD_DIR}/microbiology_cultures_priorprocedures.csv", low_memory=False).rename(columns={"order_proc_id_coded": "order_id"})
proc = proc[proc["order_id"].isin(sids)]
proc["days"] = pd.to_numeric(proc["procedure_time_to_culturetime"].replace("Null", np.nan), errors="coerce")
proc = proc[(proc["days"] > 0) & (proc["days"] <= PROC_DAYS)]
pdesc = proc["procedure_description"].astype(str).str.strip().str.lower()
for feat, key in PROC_MAP:
    stan[feat] = stan["order_id"].isin(set(proc.loc[pdesc == key, "order_id"])).astype(int)
del proc; gc.collect()

# Prior microbiology (harmonised self-join)
stan = stan.merge(build_prior_features(stan, ast_stan, orders_stan, orgs_stan), on="order_id", how="left")
stan["esbl_confirm"] = np.nan
stan = derived_features(stan)
stan = finalize_site(stan, "Stanford")
cohort_diagnostics(stan, "Stanford", t0)

Stanford feature engineering


  CLSI-organism cultures 76,244 -> urine/blood/respiratory 76,244 -> adults with known age 76,244


  Stanford comorbidity: 22,315,050 rows, 515 distinct labels; 54 labels / 7,436,884 rows map to 29 Elixhauser categories


  Stanford: 76,244 cultures, 47,082 patients | presumptive ESBL+ 6,998 (9.18%) | >=1 screening agent tested 60,294 | multi-CLSI-isolate 1,915 | label negative only because the non-susceptible isolate was not a CLSI organism: 163 | 54s


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# Optional smoke subsample, patient-level splits, Table 1
# ══════════════════════════════════════════════════════════════════════════════
if SMOKE:
    for name in ("mgb", "stan"):
        df = globals()[name]
        pids = np.array(sorted(df["patient_id"].unique()))
        keep = set(np.random.RandomState(SEED).choice(pids, max(200, int(CFG["smoke_frac"] * len(pids))), replace=False))
        globals()[name] = df[df["patient_id"].isin(keep)].reset_index(drop=True)
    mgb, stan = globals()["mgb"], globals()["stan"]
    print(f"SMOKE subsample: MGB {len(mgb):,} cultures, Stanford {len(stan):,} cultures")

for df, site in ((mgb, "MGB"), (stan, "Stanford")):
    df["split"] = make_site_splits(df)
    df[["order_id", "patient_id", "split"]].to_parquet(ckpt(f"splits_{site}.parquet"), index=False)
    counts = df.groupby("split").agg(n=("order_id", "size"), n_pos=("ESBL", "sum"), n_pat=("patient_id", "nunique"))
    rset("splits", site, value={k: {c: int(v) for c, v in row.items()} for k, row in counts.to_dict("index").items()})
    print(f"{site} splits:\n{counts.to_string()}")

# ── Table 1 ──
def _row(label, m, s, m_n=None, s_n=None):
    m_n = len(mgb) if m_n is None else m_n; s_n = len(stan) if s_n is None else s_n
    return {"label": label, "mgb_n": int(m), "mgb_pct": float(m / m_n), "stan_n": int(s), "stan_pct": float(s / s_n)}

def cohort_table(mgb, stan):
    rows = []
    def add(section, label, mcol=None, fn=None):
        m = fn(mgb).sum() if fn else mgb[mcol].sum(); s = fn(stan).sum() if fn else stan[mcol].sum()
        rows.append({"section": section, **_row(label, m, s)})
    rows.append({"section": "Overall", "label": "Total cultures", "mgb_n": len(mgb), "mgb_pct": None, "stan_n": len(stan), "stan_pct": None})
    rows.append({"section": "Overall", "label": "Patients", "mgb_n": mgb["patient_id"].nunique(), "mgb_pct": None, "stan_n": stan["patient_id"].nunique(), "stan_pct": None})
    add("Overall", "Presumptive ESBL-positive", fn=lambda d: d["ESBL"] == 1)
    add("Overall", "Presumptive ESBL-negative", fn=lambda d: d["ESBL"] == 0)
    add("Overall", r"$\geq$1 CLSI screening agent tested", fn=lambda d: d["n_screen_tested"] >= 1)
    add("Overall", "More than one CLSI-validated species isolated", fn=lambda d: d["n_clsi_isolates"] > 1)
    add("Demographics", "Male sex", fn=lambda d: d["gender_male"] == 1)
    add("Demographics", "Female sex", fn=lambda d: d["gender_male"] == 0)
    add("Demographics", r"Age $<$65 years", fn=lambda d: d["age_65plus"] == 0)
    add("Demographics", r"Age $\geq$65 years", fn=lambda d: d["age_65plus"] == 1)
    add("Demographics", "ADI high (top quintile)", "adi_high")
    add("Demographics", "ADI missing", "adi_missing")
    add("Culture type", "Urine", "cx_urine"); add("Culture type", "Blood", "cx_blood"); add("Culture type", "Respiratory", "cx_resp")
    add("Organism$^{\\S}$", r"\textit{Escherichia coli}", "org_ecoli")
    add("Organism$^{\\S}$", r"\textit{Klebsiella pneumoniae} / \textit{K. oxytoca}", "org_klebsiella")
    add("Organism$^{\\S}$", r"\textit{Proteus mirabilis}", "org_proteus")
    add("Ward setting$^{\\ddagger}$", "Outpatient", "hosp_ward_OP"); add("Ward setting$^{\\ddagger}$", "Inpatient", "hosp_ward_IP"); add("Ward setting$^{\\ddagger}$", "Emergency department", "hosp_ward_ER")
    add("Prior microbiology (3--730 days)", "Prior presumptive ESBL culture (CLSI organism)", "prior_ESBL")
    add("Prior microbiology (3--730 days)", "Prior non-susceptibility to a screening agent (any Enterobacterales)", "prior_esbl_ast")
    add("Prior microbiology (3--730 days)", "Prior carbapenem non-susceptibility", "prior_carb_resist")
    add("Prior microbiology (3--730 days)", "Any prior culture order on record", "has_prior_cx")
    for feat, lab in [("T_fq_90d","Fluoroquinolone"),("T_ceph3_90d","3rd/4th-gen cephalosporin"),("T_esp_90d","Extended-spectrum penicillin / $\\beta$-lactam combination"),
                      ("T_glyco_90d","Glycopeptide"),("T_sulfa_90d","Sulfonamide"),("T_carb_90d","Carbapenem"),("T_amino_90d","Aminoglycoside")]:
        add("Prior antibiotic exposure (90 days)", lab, feat)
    add("Comorbidities", "Renal failure / CKD", "comorbid_renal_failure"); add("Comorbidities", "Liver disease", "comorbid_liver_disease")
    add("Comorbidities", "Lymphoma", "comorbid_lymphoma"); add("Comorbidities", "Metastatic cancer", "comorbid_metastatic_cancer")
    return rows

def fmt_np(n, p): return f"{n:,}" if p is None else f"{n:,} ({p:.1%})".replace("%", r"\%")
def table1_tex(rows):
    out, sec = [], None
    for r in rows:
        if r["section"] != sec:
            sec = r["section"]; out.append(f"\\midrule\n\\textit{{{sec}}} & & \\\\")
        out.append(f"\\quad {r['label']} & {fmt_np(r['mgb_n'], r['mgb_pct'])} & {fmt_np(r['stan_n'], r['stan_pct'])} \\\\")
    return "\n".join(out) + "\n"

T1 = cohort_table(mgb, stan)
rset("table1", value=T1)
pd.DataFrame(T1).to_csv(f"{RES_DIR}/table1.csv", index=False)
write_tex("table1.tex", table1_tex(T1))
mgb_prev, stan_prev = float(mgb["ESBL"].mean()), float(stan["ESBL"].mean())
rset("cohort", "MGB", "brier_baseline", value=mgb_prev * (1 - mgb_prev)); rset("cohort", "Stanford", "brier_baseline", value=stan_prev * (1 - stan_prev))
save_results()
print(f"Table 1 written. MGB prevalence {mgb_prev:.3%} (Brier baseline {mgb_prev*(1-mgb_prev):.3f}); Stanford {stan_prev:.3%} ({stan_prev*(1-stan_prev):.3f})")

MGB splits:
             n  n_pos  n_pat
split                       
test     24063   3379  13437
train    77213  10643  42998
val_cal   9599   1274   5375
val_sel   9867   1381   5375


Stanford splits:
             n  n_pos  n_pat
split                       
test     15151   1452   9416
train    48736   4424  30132
val_cal   6134    527   3767
val_sel   6223    595   3767
  wrote cross_site_outputs/results/table1.tex
Table 1 written. MGB prevalence 13.812% (Brier baseline 0.119); Stanford 9.178% (0.083)


## Model architectures

Seven models: L1 logistic regression, XGBoost (Optuna-tuned), FT-Transformer (FTT), Multi-Head
Cross-Attention VAE (MHCA-VAE), Domain-Adaptive VAE (DA-VAE), FTT with a domain-adversarial head
(FTT-DANN) and the TabPFN v2.6 foundation model (in-context learning, no gradient training).
All architectures take the 46 harmonised features as input.

In [9]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__(); self.alpha=alpha; self.gamma=gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt = targets*torch.sigmoid(logits)+(1-targets)*(1-torch.sigmoid(logits))
        at = targets*self.alpha+(1-targets)*(1-self.alpha)
        return (at*(1-pt)**self.gamma*bce).mean()

class GroupTokenizer(nn.Module):
    def __init__(self, group_sizes, d_token):
        super().__init__()
        self.projections = nn.ModuleList([
            nn.Sequential(nn.Linear(gs, d_token), nn.LayerNorm(d_token), nn.GELU()) for gs in group_sizes])
    def forward(self, groups):
        return torch.stack([proj(g) for proj, g in zip(self.projections, groups)], dim=1)

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads; self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model); self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model); self.W_o = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        B, N, D = x.shape
        q = self.W_q(x).view(B, N, self.n_heads, self.d_k).transpose(1,2)
        k = self.W_k(x).view(B, N, self.n_heads, self.d_k).transpose(1,2)
        v = self.W_v(x).view(B, N, self.n_heads, self.d_k).transpose(1,2)
        attn = self.drop(F.softmax((q @ k.transpose(-2,-1)) / (self.d_k**0.5), dim=-1))
        out = (attn @ v).transpose(1,2).reshape(B,N,D)
        return self.W_o(out), attn

class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model); self.attn = MultiHeadCrossAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
                                nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x):
        h, attn_w = self.attn(self.ln1(x))
        x = x + h; x = x + self.ff(self.ln2(x))
        return x, attn_w

class MHCAVAE(nn.Module):
    def __init__(self, group_sizes, d_token=64, n_heads=4, n_layers=3, d_ff=256,
                 latent_dim=64, dropout=0.15, n_features_total=46):
        super().__init__()
        self.n_features = n_features_total; self.n_groups = len(group_sizes)
        self.tokenizer = GroupTokenizer(group_sizes, d_token)
        self.cls_token = nn.Parameter(torch.randn(1,1,d_token)*0.02)
        self.layers = nn.ModuleList([CrossAttentionBlock(d_token, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d_token)
        self.fc_mu = nn.Linear(d_token, latent_dim); self.fc_logvar = nn.Linear(d_token, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim,256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256,256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, n_features_total))
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim,128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64,1))
    def split_groups(self, x, group_indices): return [x[:, idx] for idx in group_indices]
    def encode(self, x, group_indices):
        groups = self.split_groups(x, group_indices)
        tokens = self.tokenizer(groups)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        attn_weights = []
        for layer in self.layers:
            tokens, aw = layer(tokens); attn_weights.append(aw)
        cls_out = self.ln(tokens[:,0])
        return self.fc_mu(cls_out), self.fc_logvar(cls_out), attn_weights
    def reparameterize(self, mu, logvar):
        if self.training: return mu + torch.randn_like(mu) * torch.exp(0.5*logvar)
        return mu
    def forward(self, x, group_indices):
        mu, logvar, attn_weights = self.encode(x, group_indices)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), self.classifier(z).squeeze(-1), mu, logvar, z, attn_weights
    def predict(self, x, group_indices):
        mu, _, _ = self.encode(x, group_indices)
        return torch.sigmoid(self.classifier(mu).squeeze(-1))

class FeatureTokenizer(nn.Module):
    def __init__(self, n_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_features, d_token)*0.02)
        self.bias = nn.Parameter(torch.zeros(n_features, d_token))
    def forward(self, x): return x.unsqueeze(-1)*self.weight.unsqueeze(0)+self.bias.unsqueeze(0)

class MHA(nn.Module):
    def __init__(self, d, n_heads, drop=0.1):
        super().__init__()
        self.n_heads=n_heads; self.d_k=d//n_heads
        self.W_q=nn.Linear(d,d); self.W_k=nn.Linear(d,d); self.W_v=nn.Linear(d,d); self.W_o=nn.Linear(d,d)
        self.drop=nn.Dropout(drop)
    def forward(self, x):
        B,N,D=x.shape
        q=self.W_q(x).view(B,N,self.n_heads,self.d_k).transpose(1,2)
        k=self.W_k(x).view(B,N,self.n_heads,self.d_k).transpose(1,2)
        v=self.W_v(x).view(B,N,self.n_heads,self.d_k).transpose(1,2)
        attn=self.drop(F.softmax((q@k.transpose(-2,-1))/(self.d_k**0.5),dim=-1))
        return self.W_o((attn@v).transpose(1,2).reshape(B,N,D))

class TBlock(nn.Module):
    def __init__(self, d, n_heads, d_ff, drop=0.1):
        super().__init__()
        self.ln1=nn.LayerNorm(d); self.attn=MHA(d,n_heads,drop); self.ln2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,d_ff),nn.GELU(),nn.Dropout(drop),nn.Linear(d_ff,d),nn.Dropout(drop))
    def forward(self, x): x=x+self.attn(self.ln1(x)); x=x+self.ff(self.ln2(x)); return x

class FTTransformer(nn.Module):
    def __init__(self, n_features, d_token=32, n_heads=4, n_layers=3, d_ff=128, dropout=0.2):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_token)
        self.cls_token = nn.Parameter(torch.randn(1,1,d_token)*0.02)
        self.layers = nn.ModuleList([TBlock(d_token, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d_token); self.head = nn.Linear(d_token, 1)
    def forward(self, x):
        tokens = torch.cat([self.cls_token.expand(x.shape[0],-1,-1), self.tokenizer(x)], dim=1)
        for layer in self.layers: tokens = layer(tokens)
        return self.head(self.ln(tokens[:,0])).squeeze(-1)

class GradRevFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha): ctx.alpha = alpha; return x.view_as(x)
    @staticmethod
    def backward(ctx, grad): return grad.neg()*ctx.alpha, None

class GRL(nn.Module):
    def __init__(self): super().__init__(); self.alpha = 1.0
    def forward(self, x): return GradRevFn.apply(x, self.alpha)

class DAVAEEncoder(nn.Module):
    def __init__(self, n_feat, hidden, latent):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_feat, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2), nn.LayerNorm(hidden//2), nn.GELU(), nn.Dropout(0.1))
        self.mu = nn.Linear(hidden//2, latent)
        self.log_var = nn.Linear(hidden//2, latent)
    def forward(self, x):
        h = self.net(x); return self.mu(h), self.log_var(h)

class DAVAEDecoder(nn.Module):
    def __init__(self, latent, hidden, n_feat):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent, hidden//2), nn.LayerNorm(hidden//2), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(hidden//2, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(hidden, n_feat))
    def forward(self, z): return self.net(z)

class DomainAdaptiveVAE(nn.Module):
    def __init__(self, n_feat, hidden=256, latent=64):
        super().__init__()
        self.encoder = DAVAEEncoder(n_feat, hidden, latent)
        self.decoder = DAVAEDecoder(latent, hidden, n_feat)
        self.classifier = nn.Sequential(
            nn.Linear(latent, latent//2), nn.GELU(), nn.Dropout(0.1), nn.Linear(latent//2, 1))
        self.grl = GRL()
        self.domain = nn.Sequential(
            nn.Linear(latent, latent//2), nn.GELU(), nn.Dropout(0.1), nn.Linear(latent//2, 1))
    def reparameterize(self, mu, lv):
        if self.training: return mu + torch.randn_like(mu) * torch.exp(0.5*lv)
        return mu
    def forward(self, x, alpha=1.0):
        mu, lv = self.encoder(x); z = self.reparameterize(mu, lv)
        self.grl.alpha = alpha
        return (self.decoder(z), self.classifier(z).squeeze(-1),
                self.domain(self.grl(z)).squeeze(-1), mu, lv, z)
    def predict(self, x):
        self.eval()
        with torch.no_grad():
            mu, lv = self.encoder(x); z = self.reparameterize(mu, lv)
            return torch.sigmoid(self.classifier(z).squeeze(-1))

class FTT_DANN(nn.Module):
    """FT-Transformer with a domain-adversarial head (gradient reversal) on the CLS token."""
    def __init__(self, n_features, d_token=32, n_heads=4, n_layers=3, d_ff=128, dropout=0.2):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_token)
        self.cls_token = nn.Parameter(torch.randn(1,1,d_token)*0.02)
        self.layers = nn.ModuleList([TBlock(d_token, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d_token)
        self.classifier = nn.Linear(d_token, 1)
        self.grl = GRL()
        self.domain_head = nn.Sequential(
            nn.Linear(d_token, d_token//2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_token//2, 1))
    def encode(self, x):
        tokens = torch.cat([self.cls_token.expand(x.shape[0],-1,-1), self.tokenizer(x)], dim=1)
        for layer in self.layers: tokens = layer(tokens)
        return self.ln(tokens[:,0])
    def forward(self, x, alpha=1.0):
        h = self.encode(x)
        self.grl.alpha = alpha
        return self.classifier(h).squeeze(-1), self.domain_head(self.grl(h)).squeeze(-1)
    def predict(self, x):
        self.eval()
        with torch.no_grad():
            return torch.sigmoid(self.classifier(self.encode(x)).squeeze(-1))

def new_ftt(n_feat=None): return FTTransformer(n_feat or N_FEAT, d_token=32, n_heads=4, n_layers=3, d_ff=128, dropout=0.2).to(DEVICE)
def new_mhcavae(): return MHCAVAE(GROUP_SIZES, d_token=64, n_heads=4, n_layers=3, d_ff=256, latent_dim=64, dropout=0.15, n_features_total=N_FEAT).to(DEVICE)
def new_davae(): return DomainAdaptiveVAE(N_FEAT).to(DEVICE)
def new_dann(): return FTT_DANN(N_FEAT).to(DEVICE)
PARAM_COUNTS = {"FTT": sum(p.numel() for p in new_ftt().parameters()), "MHCA-VAE": sum(p.numel() for p in new_mhcavae().parameters()),
                "DA-VAE": sum(p.numel() for p in new_davae().parameters()), "FTT-DANN": sum(p.numel() for p in new_dann().parameters())}
rset("param_counts", value=PARAM_COUNTS)
print("Parameter counts:", PARAM_COUNTS)

Parameter counts: {'FTT': 41185, 'MHCA-VAE': 275247, 'DA-VAE': 120368, 'FTT-DANN': 41730}


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# Training utilities, patient-cluster bootstrap, calibration and plotting helpers
# ══════════════════════════════════════════════════════════════════════════════
def s3(x):
    """Signed 3-decimal string without a negative zero (-0.0004 -> +0.000)."""
    s = f"{x:+.3f}"; return "+0.000" if s == "-0.000" else s
MODEL_KEYS = ["LR", "XGB", "FTT", "MHCA-VAE", "DA-VAE", "FTT-DANN", "TabPFN"]
MODEL_NAMES = {"LR": "LR", "XGB": "XGBoost", "FTT": "FT-Transformer", "MHCA-VAE": "MHCA-VAE", "DA-VAE": "DA-VAE",
               "FTT-DANN": "FTT-DANN", "TabPFN": "TabPFN v2.6", "XGB-warm": "XGBoost (warm start)"}
MODEL_STYLE = {"LR": ("#95a5a6", ":"), "XGB": ("#e74c3c", "--"), "FTT": ("#3498db", "-"), "MHCA-VAE": ("#9b59b6", "-"),
               "DA-VAE": ("#2ecc71", "-"), "FTT-DANN": ("#16a085", "-"), "TabPFN": ("#f39c12", "-"), "XGB-warm": ("#c0392b", "-.")}
MARKERS = {"MHCA-VAE": "o", "FTT": "s", "DA-VAE": "D", "XGB": "^", "XGB-warm": "v", "FTT-DANN": "P", "TabPFN": "*"}
DIRS = [("MGB", "Stanford"), ("Stanford", "MGB")]
def dir_tag(src, tgt): return f"{src}2{tgt}"

def site_arrays(df):
    return (df[FEATURES].to_numpy(dtype=np.float32), df["ESBL"].to_numpy(dtype=np.float32),
            df["patient_id"].astype(str).to_numpy(), df["split"].to_numpy())

def balance_cohort(X, y, seed):
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    rng = np.random.RandomState(seed)
    neg_s = rng.choice(neg, size=min(len(pos), len(neg)), replace=False)
    idx = np.concatenate([pos, neg_s]); rng.shuffle(idx)
    return X[idx], y[idx]

def make_loaders(X, y, X_val, y_val, bs=256):
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y)
    sw = np.where(y == 1, cw[1], cw[0])
    bs = min(bs, max(len(X) // 4, 16))
    trl = DataLoader(TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)),
                     batch_size=bs, sampler=WeightedRandomSampler(sw, len(sw), replacement=True), drop_last=True)
    vll = DataLoader(TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)), batch_size=2048)
    return trl, vll

def _val_auroc(pred_fn, vll):
    vp, vy = [], []
    with torch.no_grad():
        for Xv, yv in vll: vp.append(pred_fn(Xv.to(DEVICE)).cpu().numpy()); vy.append(yv.numpy())
    return roc_auc_score(np.concatenate(vy), np.concatenate(vp))

def train_ftt(model, trl, vll, epochs, lr=5e-4, pat=20, onecycle=True):
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = (optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs, steps_per_epoch=len(trl), pct_start=0.1) if onecycle
             else optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs))
    focal = FocalLoss(0.25, 2.0); best, bst, noi = 0, None, 0
    for ep in range(epochs):
        model.train()
        for X, y in trl:
            X, y = X.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); focal(model(X), y).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            if onecycle: sched.step()
        if not onecycle: sched.step()
        model.eval(); auroc = _val_auroc(lambda b: torch.sigmoid(model(b)), vll)
        if auroc > best: best, bst, noi = auroc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 0
        else: noi += 1
        if noi >= pat: break
    if bst: model.load_state_dict(bst); model.to(DEVICE)
    return model

def train_ftt_from_state(state, trl, vll, epochs, lr=1e-4, pat=10, n_feat=None):
    model = new_ftt(n_feat); model.load_state_dict(state)
    return train_ftt(model, trl, vll, epochs, lr=lr, pat=pat, onecycle=False)

def train_mhcavae(model, trl, vll, epochs, lr=5e-4, pat=20):
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    focal = FocalLoss(0.25, 2.0); gi = [torch.tensor(g) for g in GROUP_INDICES]
    best, bst, noi = 0, None, 0
    for ep in range(epochs):
        model.train()
        for X, y in trl:
            X, y = X.to(DEVICE), y.to(DEVICE); opt.zero_grad()
            x_recon, y_logit, mu, logvar, z, _ = model(X, gi)
            loss = F.mse_loss(x_recon, X) + 0.01 * (-0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())) + 5.0 * focal(y_logit, y)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sched.step()
        model.eval(); auroc = _val_auroc(lambda b: model.predict(b, gi), vll)
        if auroc > best: best, bst, noi = auroc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 0
        else: noi += 1
        if noi >= pat: break
    if bst: model.load_state_dict(bst); model.to(DEVICE)
    return model

def _davae_loop(model, X_tr, y_tr, X_val, y_val, epochs, patience, lr, bs=256):
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_tr)
    tc = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([cw[1] / cw[0]]).to(DEVICE))
    sw = np.where(y_tr == 1, cw[1], cw[0])
    loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr)), batch_size=min(bs, max(len(X_tr) // 4, 16)),
                        sampler=WeightedRandomSampler(sw, len(sw), replacement=True), drop_last=True)
    best, bst, pat = 0.0, None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for bx, by in loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE); opt.zero_grad()
            xh, yl, _, mu, lv, _ = model(bx)
            loss = F.mse_loss(xh, bx) + 0.01 * (-0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())) + 2.0 * tc(yl, by)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sched.step()
        va = roc_auc_score(y_val, pred_davae(model, X_val))
        if va > best: best, bst, pat = va, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 0
        else: pat += 1
        if pat >= patience: break
    model.load_state_dict(bst); return model

def train_davae_single(X_tr, y_tr, X_val, y_val, epochs, patience=15, lr=5e-4):
    return _davae_loop(new_davae(), X_tr, y_tr, X_val, y_val, epochs, patience, lr)
def fine_tune_davae(state, X_tr, y_tr, X_val, y_val, epochs, patience=10, lr=1e-4):
    m = new_davae(); m.load_state_dict(state); return _davae_loop(m, X_tr, y_tr, X_val, y_val, epochs, patience, lr)

def _batched(fn, X, bs=4096):
    out = []
    with torch.no_grad():
        for i in range(0, len(X), bs): out.append(fn(torch.tensor(X[i:i+bs], dtype=torch.float32).to(DEVICE)).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0, dtype=np.float32)
def pred_ftt(model, X): model.eval(); return _batched(lambda b: torch.sigmoid(model(b)), X)
def pred_mhcavae(model, X): model.eval(); gi = [torch.tensor(g) for g in GROUP_INDICES]; return _batched(lambda b: model.predict(b, gi), X)
def pred_davae(model, X): model.eval(); return _batched(lambda b: model.predict(b), X)
def pred_dann(model, X): model.eval(); return _batched(lambda b: model.predict(b), X)
def state_of(model): return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def draw_positions(tag, n, r, pool_size):
    """Positions (within the target fine-tuning pool) of the n cultures used for draw r; shared by every sweep."""
    if n >= pool_size: return np.arange(pool_size)
    return np.random.RandomState(stage_seed("draw", tag, n, r)).choice(pool_size, n, replace=False)

# ── XGBoost helpers ──
XGB_FT_PARAMS = dict(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                     scale_pos_weight=1.0, device="cuda:0", tree_method="hist", verbosity=0)
# Warm start continues boosting from the source booster; early stopping on the target validation AUROC (the same
# criterion the neural models use), because log-loss on a natural-prevalence validation set is dominated by the
# prevalence mismatch of the class-balanced fine-tuning subsample and halts boosting immediately.
XGB_WARM_PARAMS = dict(n_estimators=200, max_depth=6, learning_rate=0.02, subsample=0.8, colsample_bytree=0.8,
                       scale_pos_weight=1.0, device="cuda:0", tree_method="hist", verbosity=0, early_stopping_rounds=20, eval_metric="auc")
def xgb_predict(model, X): return model.predict_proba(X)[:, 1]

# ── Patient-cluster bootstrap ──
_W_CACHE = {}
def boot_weights(pids, nb=None, seed=SEED, key=None):
    """nb x n matrix of patient-draw multiplicities (uint8). Cached by key so all models on the same
    evaluation set share identical resamples (paired comparisons)."""
    nb = nb or CFG["n_boot"]
    if key is not None and key in _W_CACHE: return _W_CACHE[key]
    uniq, inv = np.unique(np.asarray(pids).astype(str), return_inverse=True)
    G = len(uniq); rng = np.random.RandomState(seed)
    W = np.empty((nb, len(inv)), dtype=np.uint8)
    for b in range(nb):
        W[b] = np.minimum(np.bincount(rng.randint(0, G, G), minlength=G)[inv], 255)
    if key is not None: _W_CACHE[key] = W
    return W

class Presort:
    """Weighted rank AUROC / AP / Brier for one (y, p) pair; multiplicity weights = resampled patients."""
    def __init__(self, y, p):
        y = np.asarray(y, dtype=np.float64); p = np.asarray(p, dtype=np.float64)
        self.order = np.argsort(p, kind="mergesort"); ps = p[self.order]
        self.ys = y[self.order]; self.p = p; self.y = y
        brk = np.flatnonzero(np.diff(ps) != 0) + 1
        self.starts = np.r_[0, brk]; self.ends = np.r_[brk, len(ps)]
    def auroc(self, w):
        ws = w[self.order].astype(np.float64)
        cw = np.cumsum(ws); pw = ws * self.ys; cpw = np.cumsum(pw)
        before = np.where(self.starts > 0, cw[self.starts - 1], 0.0)
        Wg = cw[self.ends - 1] - before
        PWg = cpw[self.ends - 1] - np.where(self.starts > 0, cpw[self.starts - 1], 0.0)
        n_pos = pw.sum(); n_neg = ws.sum() - n_pos
        if n_pos == 0 or n_neg == 0: return np.nan
        R = ((before + (Wg + 1) / 2.0) * PWg).sum()
        return (R - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    def ap(self, w):
        ws = w[self.order].astype(np.float64)[::-1]; ys = self.ys[::-1]; n = len(ws)
        starts = n - self.ends[::-1]; ends = n - self.starts[::-1]
        cw = np.cumsum(ws); cpw = np.cumsum(ws * ys)
        if cpw[-1] == 0: return np.nan
        tp = cpw[ends - 1]; tot = cw[ends - 1]
        prec = tp / np.maximum(tot, 1e-12); rec = tp / cpw[-1]
        return float(np.sum((rec - np.r_[0.0, rec[:-1]]) * prec))
    def brier(self, w):
        w = w.astype(np.float64); return float(np.sum(w * (self.p - self.y) ** 2) / np.sum(w))

def point_metrics(y, p):
    return {"auroc": float(roc_auc_score(y, p)), "auprc": float(average_precision_score(y, p)), "brier": float(brier_score_loss(y, p))}

def boot_metrics(y, p, W, which=("auroc", "auprc", "brier")):
    ps = Presort(y, p); out = {}
    if "auroc" in which: out["auroc"] = np.array([ps.auroc(w) for w in W])
    if "auprc" in which: out["auprc"] = np.array([ps.ap(w) for w in W])
    if "brier" in which: out["brier"] = np.array([ps.brier(w) for w in W])
    return out

def ci(a): a = np.asarray(a, dtype=np.float64); a = a[~np.isnan(a)]; return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]

def eval_with_ci(y, p, W, which=("auroc", "auprc", "brier")):
    m = point_metrics(y, p); b = boot_metrics(y, p, W, which)
    for k in which: m[f"ci_{k}"] = ci(b[k])
    return m

def paired_delta(y, pA, pB, W, metric="auroc"):
    a, b = Presort(y, pA), Presort(y, pB)
    fa = a.auroc if metric == "auroc" else a.ap; fb = b.auroc if metric == "auroc" else b.ap
    return np.array([fa(w) - fb(w) for w in W])

def weighted_confusion(y, p, thr, w):
    pred = (p >= thr); w = w.astype(np.float64)
    tp = float(np.sum(w * (pred & (y == 1)))); fp = float(np.sum(w * (pred & (y == 0))))
    fn = float(np.sum(w * (~pred & (y == 1)))); tn = float(np.sum(w * (~pred & (y == 0))))
    return {"sens": tp / max(tp + fn, 1e-12), "spec": tn / max(tn + fp, 1e-12), "ppv": tp / max(tp + fp, 1e-12), "npv": tn / max(tn + fn, 1e-12),
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}

# ── Calibration helpers ──
def fit_isotonic(p_val, y_val):
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0); iso.fit(p_val, y_val.astype(int)); return iso
def _logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def fit_intercept_recal(p_val, y_val):
    """Intercept-only logistic recalibration: p' = sigmoid(logit(p) + a), a fitted by maximum likelihood."""
    from scipy.optimize import minimize_scalar
    z = _logit(p_val); y = y_val.astype(np.float64)
    def nll(a): q = 1 / (1 + np.exp(-(z + a))); q = np.clip(q, 1e-9, 1 - 1e-9); return -np.mean(y * np.log(q) + (1 - y) * np.log(1 - q))
    a = minimize_scalar(nll, bounds=(-10, 10), method="bounded").x
    return lambda p: 1 / (1 + np.exp(-(_logit(p) + a)))
def calib_slope_intercept(y, p):
    lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000).fit(_logit(p).reshape(-1, 1), y.astype(int))
    return float(lr.coef_[0][0]), float(lr.intercept_[0])
def reliability_bins(y, p, n_bins=10, strategy="quantile"):
    if strategy == "quantile":
        edges = np.unique(np.quantile(p, np.linspace(0, 1, n_bins + 1))); edges[-1] = np.nextafter(edges[-1], np.inf)
    else: edges = np.linspace(0, 1, n_bins + 1); edges[-1] = 1.0001
    b = np.clip(np.digitize(p, edges[1:-1]), 0, len(edges) - 2)
    rows = []
    for k in range(len(edges) - 1):
        m = b == k
        if m.sum() == 0: continue
        rows.append({"bin": k, "n": int(m.sum()), "mean_pred": float(p[m].mean()), "frac_pos": float(y[m].mean())})
    return pd.DataFrame(rows)
def ece(y, p, n_bins=10, strategy="quantile"):
    rb = reliability_bins(y, p, n_bins, strategy); return float(np.sum(rb["n"] / rb["n"].sum() * np.abs(rb["frac_pos"] - rb["mean_pred"])))
def calibration_summary(y, p):
    s, i = calib_slope_intercept(y, p)
    return {"brier": float(brier_score_loss(y, p)), "ece": ece(y, p), "slope": s, "intercept": i, "mean_pred": float(np.mean(p))}

# ── Plot helpers ──
def plot_roc(ax, y, preds):
    for key, p in preds.items():
        c, ls = MODEL_STYLE[key]; fpr, tpr, _ = roc_curve(y, p)
        ax.plot(fpr, tpr, color=c, linestyle=ls, lw=2.2, label=f"{MODEL_NAMES[key]} ({roc_auc_score(y, p):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("1 - Specificity"); ax.set_ylabel("Sensitivity"); ax.legend(loc="lower right", fontsize=9)
def plot_pr_step(ax, y, preds):
    """Precision-recall as a step function over observed thresholds; sklearn's synthetic (recall 0, precision 1) end point is dropped."""
    for key, p in preds.items():
        c, ls = MODEL_STYLE[key]; prec, rec, _ = precision_recall_curve(y, p)
        prec, rec = prec[:-1], rec[:-1]
        ax.step(rec, prec, where="post", color=c, linestyle=ls, lw=2.2, label=f"{MODEL_NAMES[key]} ({average_precision_score(y, p):.3f})")
    ax.axhline(y.mean(), color="k", linestyle="--", lw=1, label=f"Prevalence ({y.mean():.3f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(loc="upper right", fontsize=9)
def plot_reliability(ax, y, preds, n_bins=10, min_n=30, annotate=True):
    for key, p in preds.items():
        c, ls = MODEL_STYLE[key]; rb = reliability_bins(y, p, n_bins, "quantile"); ok = rb["n"] >= min_n
        ax.plot(rb.loc[ok, "mean_pred"], rb.loc[ok, "frac_pos"], "s-", color=c, linestyle=ls, lw=1.8, ms=5,
                label=f"{MODEL_NAMES[key]} (Brier {brier_score_loss(y, p):.3f}, ECE {ece(y, p):.3f})")
        if (~ok).any(): ax.plot(rb.loc[~ok, "mean_pred"], rb.loc[~ok, "frac_pos"], "s", color=c, ms=5, mfc="none")
        if annotate:
            for _, r in rb.iterrows(): ax.annotate(f"{int(r['n'])}", (r["mean_pred"], r["frac_pos"]), fontsize=5, color=c, xytext=(2, 2), textcoords="offset points")
    ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Mean predicted probability (quantile bins)"); ax.set_ylabel("Observed proportion"); ax.legend(loc="upper left", fontsize=8)
def dca_curve(y, p, thresholds):
    N = len(y); out = []
    for t in thresholds:
        tp = np.sum((p >= t) & (y == 1)); fp = np.sum((p >= t) & (y == 0)); out.append(tp / N - fp / N * (t / (1 - t)))
    return np.array(out)
def plot_dca(ax, y, preds, tmax=0.40):
    th = np.linspace(0.01, tmax, 60)
    for key, p in preds.items():
        c, ls = MODEL_STYLE[key]; ax.plot(th, dca_curve(y, p, th), color=c, linestyle=ls, lw=2.2, label=MODEL_NAMES[key])
    ax.plot(th, y.mean() - (1 - y.mean()) * th / (1 - th), "k:", lw=2, label="Treat all"); ax.axhline(0, color="k", lw=2, label="Treat none")
    ax.set_ylim(-0.02, max(0.12, y.mean() + 0.01)); ax.set_xlabel("Threshold probability"); ax.set_ylabel("Net benefit"); ax.legend(loc="upper right", fontsize=9)
def four_panel_figure(y, raw, cal, title, path):
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    plot_roc(axes[0, 0], y, raw); axes[0, 0].set_title("A. ROC curve (raw scores)", fontweight="bold", fontsize=14)
    plot_pr_step(axes[0, 1], y, raw); axes[0, 1].set_title("B. Precision-recall curve (raw scores)", fontweight="bold", fontsize=14)
    plot_reliability(axes[1, 0], y, cal); axes[1, 0].set_title("C. Calibration (isotonic, fitted on source calibration split)", fontweight="bold", fontsize=13)
    plot_dca(axes[1, 1], y, cal); axes[1, 1].set_title("D. Decision curve analysis (calibrated probabilities)", fontweight="bold", fontsize=13)
    plt.suptitle(title, fontsize=16, fontweight="bold"); plt.tight_layout(); plt.savefig(path, dpi=300); plt.close(fig)
    print(f"  saved {path}")
print("utils ready")

utils ready


## Zero-shot bidirectional external validation

For each direction the source site's `train` split (64 % of patients) trains the model, `val_sel` (8 %) drives
early stopping and Optuna, `val_cal` (8 %) fits the isotonic calibrator and the operating threshold, and
`test` (20 %) is held out for permutation importance and source-site forgetting. The full target site is the
external test set. Raw and isotonic-calibrated predictions are saved for every model.

In [11]:
def scaler_from(train_X):
    sc = StandardScaler().fit(train_X); return sc
def save_scaler(name, sc): np.savez(ckpt(name), mean=sc.mean_, scale=sc.scale_)
def load_scaler(name):
    z = np.load(ckpt(name)); sc = StandardScaler(); sc.mean_ = z["mean"]; sc.scale_ = z["scale"]; sc.var_ = sc.scale_ ** 2
    sc.n_features_in_ = len(sc.mean_); return sc

def run_experiment(source_df, target_df, src_name, tgt_name):
    tag = dir_tag(src_name, tgt_name); t0 = time.time()
    print(f"\n{'='*70}\nZERO-SHOT: {src_name} -> {tgt_name}\n{'='*70}")
    Xs, ys, ps, ss = site_arrays(source_df); Xt, yt, pt, _ = site_arrays(target_df)
    tr, vs, vc, te = (ss == "train"), (ss == "val_sel"), (ss == "val_cal"), (ss == "test")
    done = ckpt(f"{tag}_zero_shot_done.json")
    if RESUME and os.path.exists(done):
        print("  resuming from checkpoint"); scaler = load_scaler(f"{tag}_scaler.npz")
        states = {m: load_state(f"{tag}_{m}.pt") for m in ("FTT", "MHCA-VAE", "DA-VAE")}
        xgb_m = XGBClassifier(); xgb_m.load_model(ckpt(f"{tag}_XGB.ubj"))
        lr_m = pickle.load(open(ckpt(f"{tag}_LR.pkl"), "rb"))
        preds = {m: load_preds(f"zs_{tag}_{m}") for m in ("LR", "XGB", "FTT", "MHCA-VAE", "DA-VAE")}
        return {"tag": tag, "scaler": scaler, "states": states, "xgb": xgb_m, "lr": lr_m, "preds": preds,
                "xgb_params": json.load(open(done))["xgb_params"]}
    scaler = scaler_from(Xs[tr]); save_scaler(f"{tag}_scaler.npz", scaler)
    Xs_ = scaler.transform(Xs).astype(np.float32); Xt_ = scaler.transform(Xt).astype(np.float32)
    X_tr, y_tr = Xs_[tr], ys[tr]; X_vs, y_vs = Xs_[vs], ys[vs]; X_vc, y_vc = Xs_[vc], ys[vc]; X_te = Xs_[te]
    X_tr_bal, y_tr_bal = balance_cohort(X_tr, y_tr, stage_seed("balance", tag))
    trl, vll = make_loaders(X_tr, y_tr, X_vs, y_vs)
    print(f"  source train {tr.sum():,} (balanced {len(X_tr_bal):,}) | val_sel {vs.sum():,} | val_cal {vc.sum():,} | test {te.sum():,} | target {len(Xt_):,}")
    preds, states = {}, {}
    def finish(key, model_pred):
        p_t, p_te, p_vc = model_pred(Xt_), model_pred(X_te), model_pred(X_vc)
        iso = fit_isotonic(p_vc, y_vc); pickle.dump(iso, open(ckpt(f"{tag}_{key}_iso.pkl"), "wb"))
        preds[key] = {"target_raw": p_t, "srctest_raw": p_te, "valcal_raw": p_vc,
                      "target_cal": iso.transform(p_t), "srctest_cal": iso.transform(p_te)}
        save_preds(f"zs_{tag}_{key}", **preds[key])
        print(f"  {MODEL_NAMES[key]:<16s} target AUROC {roc_auc_score(yt, p_t):.4f} | source-test AUROC {roc_auc_score(ys[te], p_te):.4f} | {time.time()-t0:.0f}s")

    # 1. Logistic regression (L1) on the balanced cohort
    lr_m = LogisticRegression(penalty="l1", solver="saga", max_iter=300, random_state=SEED).fit(X_tr_bal, y_tr_bal)
    pickle.dump(lr_m, open(ckpt(f"{tag}_LR.pkl"), "wb")); finish("LR", lambda X: lr_m.predict_proba(X)[:, 1])

    # 2. XGBoost tuned with Optuna on val_sel
    def obj(trial):
        p = {"max_depth": trial.suggest_int("max_depth", 3, 10), "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
             "n_estimators": trial.suggest_int("n_estimators", 100, 600, step=50), "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
             "subsample": trial.suggest_float("subsample", 0.5, 1.0), "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
             "gamma": trial.suggest_float("gamma", 0, 5), "eval_metric": "logloss", "random_state": SEED, "verbosity": 0,
             "device": "cuda:0", "tree_method": "hist"}
        return roc_auc_score(y_vs, XGBClassifier(**p).fit(X_tr_bal, y_tr_bal).predict_proba(X_vs)[:, 1])
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(obj, n_trials=CFG["optuna_trials"], show_progress_bar=False)
    bp = dict(study.best_params); bp.update({"eval_metric": "logloss", "random_state": SEED, "verbosity": 0, "device": "cuda:0", "tree_method": "hist"})
    xgb_m = XGBClassifier(**bp).fit(X_tr_bal, y_tr_bal); xgb_m.save_model(ckpt(f"{tag}_XGB.ubj"))
    finish("XGB", lambda X: xgb_m.predict_proba(X)[:, 1])

    # 3. FT-Transformer
    seed_everything(stage_seed("zs", tag, "FTT")); torch.cuda.empty_cache()
    ftt = train_ftt(new_ftt(), trl, vll, CFG["epochs"], lr=5e-4, pat=CFG["patience"])
    states["FTT"] = state_of(ftt); save_state(f"{tag}_FTT.pt", states["FTT"]); finish("FTT", lambda X: pred_ftt(ftt, X)); del ftt

    # 4. MHCA-VAE
    seed_everything(stage_seed("zs", tag, "MHCA-VAE")); torch.cuda.empty_cache()
    vae = train_mhcavae(new_mhcavae(), trl, vll, CFG["epochs"], lr=5e-4, pat=CFG["patience"])
    states["MHCA-VAE"] = state_of(vae); save_state(f"{tag}_MHCA-VAE.pt", states["MHCA-VAE"]); finish("MHCA-VAE", lambda X: pred_mhcavae(vae, X)); del vae

    # 5. DA-VAE (single-site training: reconstruction + KL + classifier)
    seed_everything(stage_seed("zs", tag, "DA-VAE")); torch.cuda.empty_cache()
    dav = train_davae_single(X_tr, y_tr, X_vs, y_vs, CFG["epochs"] if not SMOKE else 2, patience=15)
    states["DA-VAE"] = state_of(dav); save_state(f"{tag}_DA-VAE.pt", states["DA-VAE"]); finish("DA-VAE", lambda X: pred_davae(dav, X)); del dav
    torch.cuda.empty_cache(); gc.collect()
    json.dump({"xgb_params": bp, "seconds": time.time() - t0}, open(done, "w"))
    return {"tag": tag, "scaler": scaler, "states": states, "xgb": xgb_m, "lr": lr_m, "preds": preds, "xgb_params": bp}

In [12]:
t_zs = time.time()
EXP = {}
EXP["MGB2Stanford"] = run_experiment(mgb, stan, "MGB", "Stanford")
EXP["Stanford2MGB"] = run_experiment(stan, mgb, "Stanford", "MGB")
SITE_DF = {"MGB": mgb, "Stanford": stan}
for tag, e in EXP.items(): rset("xgb_params", tag, value=e["xgb_params"])
save_results()
print(f"\nZero-shot experiments done in {(time.time()-t_zs)/60:.1f} min")


ZERO-SHOT: MGB -> Stanford


  source train 77,213 (balanced 21,286) | val_sel 9,867 | val_cal 9,599 | test 24,063 | target 76,244


  LR               target AUROC 0.7176 | source-test AUROC 0.7762 | 2s


  XGBoost          target AUROC 0.7266 | source-test AUROC 0.7908 | 12s


  FT-Transformer   target AUROC 0.7344 | source-test AUROC 0.7879 | 194s


...

  LR               target AUROC 0.7465 | source-test AUROC 0.7566 | 0s


  XGBoost          target AUROC 0.7474 | source-test AUROC 0.7596 | 7s


  FT-Transformer   target AUROC 0.7516 | source-test AUROC 0.7627 | 90s


  MHCA-VAE         target AUROC 0.7517 | source-test AUROC 0.7596 | 176s


  DA-VAE           target AUROC 0.7479 | source-test AUROC 0.7546 | 194s

Zero-shot experiments done in 10.0 min


## TabPFN v2.6 (in-context learning, no gradient training)

The context is a random natural-prevalence sample of the source `train` split (10,000 cultures), so the
predicted probabilities are on the natural-prevalence scale. Requires the `tabpfn` package and a PriorLabs
licence token in `TABPFN_TOKEN` (read from the environment or `.env`); the cell skips gracefully otherwise.

In [13]:
TABPFN_AVAILABLE = False
try:
    import tabpfn
    from tabpfn import TabPFNClassifier
    from tabpfn.constants import ModelVersion
    TABPFN_AVAILABLE = True
    print(f"tabpfn {tabpfn.__version__} imported; token present: {bool(os.environ.get('TABPFN_TOKEN'))}")
except Exception as e:
    print(f"TabPFN unavailable ({type(e).__name__}: {e}); TabPFN analyses will be skipped")

def new_tabpfn(seed=SEED):
    return TabPFNClassifier.create_default_for_version(ModelVersion.V2_6, device="cuda", random_state=seed)

def run_tabpfn(source_df, target_df, src_name, tgt_name):
    tag = dir_tag(src_name, tgt_name); key = "TabPFN"; t0 = time.time()
    if have_preds(f"zs_{tag}_{key}"):
        print(f"  TabPFN {tag}: loaded from disk"); return load_preds(f"zs_{tag}_{key}")
    Xs, ys, ps, ss = site_arrays(source_df); Xt, yt, pt, _ = site_arrays(target_df)
    tr_idx = np.flatnonzero(ss == "train"); rng = np.random.RandomState(stage_seed("tabpfn", tag))
    ctx = rng.choice(tr_idx, min(CFG["tabpfn_context"], len(tr_idx)), replace=False)
    clf = new_tabpfn(); clf.fit(Xs[ctx], ys[ctx].astype(int))
    p_t = clf.predict_proba(Xt)[:, 1]; p_te = clf.predict_proba(Xs[ss == "test"])[:, 1]; p_vc = clf.predict_proba(Xs[ss == "val_cal"])[:, 1]
    iso = fit_isotonic(p_vc, ys[ss == "val_cal"]); pickle.dump(iso, open(ckpt(f"{tag}_{key}_iso.pkl"), "wb"))
    out = {"target_raw": p_t, "srctest_raw": p_te, "valcal_raw": p_vc, "target_cal": iso.transform(p_t), "srctest_cal": iso.transform(p_te)}
    save_preds(f"zs_{tag}_{key}", **out)
    print(f"  TabPFN {tag}: context {len(ctx):,} ({ys[ctx].mean():.1%} ESBL+) | target AUROC {roc_auc_score(yt, p_t):.4f} | {time.time()-t0:.0f}s")
    return out

def run_tabpfn_sweep(source_df, target_df, src_name, tgt_name):
    """Data-efficiency analogue for TabPFN: replace part of the source context with n target fine-tuning cultures
    (draw 0 of the main sweep). Evaluated on the target test split and the source test split."""
    tag = dir_tag(src_name, tgt_name); path = f"{RES_DIR}/tabpfn_sweep_{tag}.json"
    if RESUME and os.path.exists(path): return json.load(open(path))
    Xs, ys, ps, ss = site_arrays(source_df); Xt, yt, pt, st = site_arrays(target_df)
    src_tr = np.flatnonzero(ss == "train"); src_te = ss == "test"; tgt_pool = np.flatnonzero(st == "train"); tgt_te = st == "test"
    rng = np.random.RandomState(stage_seed("tabpfn", tag))
    ctx = rng.choice(src_tr, min(CFG["tabpfn_context"], len(src_tr)), replace=False)
    rows = []
    for n in [0] + CFG["budgets"]:
        n = min(n, len(tgt_pool))
        if n > 0:
            sub = tgt_pool[draw_positions(tag, n, 0, len(tgt_pool))]
            keep = rng.choice(ctx, max(len(ctx) - n, 0), replace=False)
            Xc = np.concatenate([Xs[keep], Xt[sub]]); yc = np.concatenate([ys[keep], yt[sub]])
        else: Xc, yc = Xs[ctx], ys[ctx]
        clf = new_tabpfn(); clf.fit(Xc, yc.astype(int))
        p_t = clf.predict_proba(Xt[tgt_te])[:, 1]; p_s = clf.predict_proba(Xs[src_te])[:, 1]
        rows.append({"n_ft": int(n), "auroc_tgt": float(roc_auc_score(yt[tgt_te], p_t)), "auprc_tgt": float(average_precision_score(yt[tgt_te], p_t)),
                     "auroc_src": float(roc_auc_score(ys[src_te], p_s))})
        print(f"  TabPFN sweep {tag} n={n:,}: target-test AUROC {rows[-1]['auroc_tgt']:.4f} | source-test {rows[-1]['auroc_src']:.4f}")
    json.dump(rows, open(path, "w")); return rows

if TABPFN_AVAILABLE:
    try:
        for src, tgt in DIRS:
            EXP[dir_tag(src, tgt)]["preds"]["TabPFN"] = run_tabpfn(SITE_DF[src], SITE_DF[tgt], src, tgt)
        TABPFN_SWEEP = {dir_tag("MGB", "Stanford"): run_tabpfn_sweep(mgb, stan, "MGB", "Stanford")}
        rset("tabpfn", "version", value=tabpfn.__version__); rset("tabpfn", "model_version", value="V2_6"); rset("tabpfn", "sweep", value=TABPFN_SWEEP)
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"TabPFN failed ({type(e).__name__}: {e}); continuing without TabPFN")
        TABPFN_AVAILABLE = False
if not TABPFN_AVAILABLE:
    TABPFN_SWEEP = {}
    for tag in EXP: EXP[tag]["preds"].pop("TabPFN", None)
rset("tabpfn", "available", value=TABPFN_AVAILABLE); save_results()

tabpfn 8.0.7 imported; token present: True


  TabPFN MGB2Stanford: context 10,000 (13.8% ESBL+) | target AUROC 0.7321 | 107s


  TabPFN Stanford2MGB: context 10,000 (9.6% ESBL+) | target AUROC 0.7411 | 140s


  TabPFN sweep MGB2Stanford n=0: target-test AUROC 0.7314 | source-test 0.7881


  TabPFN sweep MGB2Stanford n=500: target-test AUROC 0.7297 | source-test 0.7883


  TabPFN sweep MGB2Stanford n=1,000: target-test AUROC 0.7425 | source-test 0.7884


  TabPFN sweep MGB2Stanford n=2,000: target-test AUROC 0.7407 | source-test 0.7885


  TabPFN sweep MGB2Stanford n=5,000: target-test AUROC 0.7450 | source-test 0.7853


## FTT-DANN and the seed study

FTT-DANN is trained on labelled source `train` cultures plus **unlabelled target cultures from the target
training pool only** (the 20 % target test split is never seen). Zero-shot FTT-DANN and plain FTT are each
trained with five seeds per direction to quantify initialisation variance; the run with seed 42 is the
point estimate used in Table 2.

In [14]:
import itertools

def train_fttdann_joint(X_src, y_src, X_tgt_unl, X_val, y_val, epochs, patience=15, lr=5e-4, lam=0.5):
    torch.cuda.empty_cache(); gc.collect()
    model = new_dann()
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    focal = FocalLoss(0.25, 2.0); dc = nn.BCEWithLogitsLoss()
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_src)
    sw = np.where(y_src == 1, cw[1], cw[0])
    src_loader = DataLoader(TensorDataset(torch.FloatTensor(X_src), torch.FloatTensor(y_src), torch.zeros(len(X_src))), batch_size=256,
                            sampler=WeightedRandomSampler(sw, len(sw), replacement=True), drop_last=True)
    tgt_loader = DataLoader(TensorDataset(torch.FloatTensor(X_tgt_unl), torch.ones(len(X_tgt_unl))), batch_size=256, shuffle=True, drop_last=True)
    best, bst, pat = 0.0, None, 0
    for ep in range(1, epochs + 1):
        model.train(); alpha = float(2.0 / (1.0 + np.exp(-10 * ep / epochs)) - 1); tgt_iter = itertools.cycle(tgt_loader)
        for (bx_s, by_s, bd_s) in src_loader:
            (bx_t, bd_t) = next(tgt_iter)
            bx_s, by_s, bd_s = bx_s.to(DEVICE), by_s.to(DEVICE), bd_s.to(DEVICE); bx_t, bd_t = bx_t.to(DEVICE), bd_t.to(DEVICE)
            opt.zero_grad()
            yl_s, dl_s = model(bx_s, alpha=alpha); _, dl_t = model(bx_t, alpha=alpha)
            loss = focal(yl_s, by_s) + lam * (dc(dl_s, bd_s) + dc(dl_t, bd_t))
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sched.step()
        va = roc_auc_score(y_val, pred_dann(model, X_val))
        if va > best: best, bst, pat = va, state_of(model), 0
        else: pat += 1
        if pat >= patience: break
    model.load_state_dict(bst); return model

def fine_tune_fttdann(state, X_tr, y_tr, X_val, y_val, epochs, patience=10, lr=1e-4):
    torch.cuda.empty_cache(); gc.collect()
    model = new_dann(); model.load_state_dict(state)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    focal = FocalLoss(0.25, 2.0)
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_tr); sw = np.where(y_tr == 1, cw[1], cw[0])
    loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr)), batch_size=min(256, max(len(X_tr) // 4, 16)),
                        sampler=WeightedRandomSampler(sw, len(sw), replacement=True), drop_last=True)
    best, bst, pat = 0.0, None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for bx, by in loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE); opt.zero_grad()
            yl, _ = model(bx, alpha=0.0); focal(yl, by).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sched.step()
        va = roc_auc_score(y_val, pred_dann(model, X_val))
        if va > best: best, bst, pat = va, state_of(model), 0
        else: pat += 1
        if pat >= patience: break
    model.load_state_dict(bst); return model

PRIMARY_SEED = SEED if SEED in CFG["seeds"] else CFG["seeds"][0]
SEED_STUDY = {}
t_seed = time.time()
for src, tgt in DIRS:
    tag = dir_tag(src, tgt); scaler = EXP[tag]["scaler"]
    Xs, ys, ps, ss = site_arrays(SITE_DF[src]); Xt, yt, pt, st = site_arrays(SITE_DF[tgt])
    Xs_ = scaler.transform(Xs).astype(np.float32); Xt_ = scaler.transform(Xt).astype(np.float32)
    tr, vs, vc, te = (ss == "train"), (ss == "val_sel"), (ss == "val_cal"), (ss == "test")
    X_unl = Xt_[st != "test"]      # target training pool only (no target test data)
    trl, vll = make_loaders(Xs_[tr], ys[tr], Xs_[vs], ys[vs])
    SEED_STUDY[tag] = {"FTT-DANN": {}, "FTT": {}}
    for seed in CFG["seeds"]:
        for key in ("FTT-DANN", "FTT"):
            name = f"seed_{tag}_{key}_{seed}"
            if have_preds(name):
                z = load_preds(name); p_t = z["target_raw"]
                if key == "FTT-DANN" and seed == PRIMARY_SEED:
                    EXP[tag]["states"]["FTT-DANN"] = load_state(f"{tag}_FTT-DANN.pt"); EXP[tag]["preds"]["FTT-DANN"] = load_preds(f"zs_{tag}_FTT-DANN")
            else:
                seed_everything(stage_seed("seedstudy", tag, key, seed)); t0 = time.time()
                if key == "FTT-DANN":
                    m = train_fttdann_joint(Xs_[tr], ys[tr], X_unl, Xs_[vs], ys[vs], CFG["epochs"] if not SMOKE else 2, patience=15)
                    pred = lambda X: pred_dann(m, X)
                else:
                    m = train_ftt(new_ftt(), trl, vll, CFG["epochs"], lr=5e-4, pat=CFG["patience"]); pred = lambda X: pred_ftt(m, X)
                p_t = pred(Xt_); save_preds(name, target_raw=p_t)
                if key == "FTT-DANN" and seed == PRIMARY_SEED:
                    p_te, p_vc = pred(Xs_[te]), pred(Xs_[vc]); iso = fit_isotonic(p_vc, ys[vc])
                    pickle.dump(iso, open(ckpt(f"{tag}_FTT-DANN_iso.pkl"), "wb"))
                    EXP[tag]["preds"]["FTT-DANN"] = {"target_raw": p_t, "srctest_raw": p_te, "valcal_raw": p_vc, "target_cal": iso.transform(p_t), "srctest_cal": iso.transform(p_te)}
                    save_preds(f"zs_{tag}_FTT-DANN", **EXP[tag]["preds"]["FTT-DANN"])
                    EXP[tag]["states"]["FTT-DANN"] = state_of(m); save_state(f"{tag}_FTT-DANN.pt", EXP[tag]["states"]["FTT-DANN"])
                print(f"  {tag} {key} seed {seed}: target AUROC {roc_auc_score(yt, p_t):.4f} ({time.time()-t0:.0f}s)")
                del m; torch.cuda.empty_cache(); gc.collect()
            SEED_STUDY[tag][key][seed] = float(roc_auc_score(yt, p_t))
    for key in ("FTT-DANN", "FTT"):
        v = np.array(list(SEED_STUDY[tag][key].values()))
        rset("seed_study", tag, key, value={"per_seed": SEED_STUDY[tag][key], "mean": float(v.mean()), "sd": float(v.std(ddof=1)) if len(v) > 1 else 0.0,
                                            "min": float(v.min()), "max": float(v.max()), "primary_seed": PRIMARY_SEED})
        print(f"  {tag} {key}: seeds {sorted(SEED_STUDY[tag][key])} -> mean {v.mean():.4f} SD {v.std(ddof=1) if len(v)>1 else 0:.4f} range [{v.min():.4f}, {v.max():.4f}]")
save_results(); print(f"Seed study done in {(time.time()-t_seed)/60:.1f} min")

  MGB2Stanford FTT-DANN seed 0: target AUROC 0.7418 (357s)


  MGB2Stanford FTT seed 0: target AUROC 0.7428 (156s)


  MGB2Stanford FTT-DANN seed 7: target AUROC 0.7339 (265s)


  MGB2Stanford FTT seed 7: target AUROC 0.7362 (155s)


  MGB2Stanford FTT-DANN seed 42: target AUROC 0.7402 (339s)


...

  Stanford2MGB FTT seed 42: target AUROC 0.7525 (84s)


  Stanford2MGB FTT-DANN seed 123: target AUROC 0.7405 (119s)


  Stanford2MGB FTT seed 123: target AUROC 0.7592 (110s)


  Stanford2MGB FTT-DANN seed 2024: target AUROC 0.7519 (178s)


  Stanford2MGB FTT seed 2024: target AUROC 0.7567 (81s)
  Stanford2MGB FTT-DANN: seeds [0, 7, 42, 123, 2024] -> mean 0.7463 SD 0.0045 range [0.7405, 0.7519]
  Stanford2MGB FTT: seeds [0, 7, 42, 123, 2024] -> mean 0.7543 SD 0.0038 range [0.7494, 0.7592]
Seed study done in 61.4 min


## Transfer data-efficiency sweep

Target-site cultures are split by patient with the same 64 / 8 / 8 / 20 rule: the 64 % `train` split is the
fine-tuning pool, `val_sel` is the fine-tuning validation split (early stopping, warm-start early stopping,
local re-thresholding), `val_cal` is reserved for target-site recalibration, and `test` (20 %) is the target
test split. For each budget n in {500, 1,000, 2,000, 5,000} ten random subsamples are drawn from the pool
(draw 0 is the pre-declared primary draw); the ALL budget is repeated with three seeds. Each draw fine-tunes
MHCA-VAE, FT-Transformer, DA-VAE (and FTT-DANN for MGB→Stanford) from the source-trained weights and trains
XGBoost from scratch and warm-started from the source booster. Source-site forgetting is measured on the
source `test` split. Results are appended to a JSONL file per direction so the stage can resume.

In [15]:
def prepare_transfer(src_name, tgt_name):
    tag = dir_tag(src_name, tgt_name); scaler = EXP[tag]["scaler"]
    Xs, ys, ps, ss = site_arrays(SITE_DF[src_name]); Xt, yt, pt, st = site_arrays(SITE_DF[tgt_name])
    Xs_ = scaler.transform(Xs).astype(np.float32); Xt_ = scaler.transform(Xt).astype(np.float32)
    pool, vs, vc, te = (st == "train"), (st == "val_sel"), (st == "val_cal"), (st == "test")
    return {"tag": tag, "X_pool": Xt_[pool], "y_pool": yt[pool], "X_val": Xt_[vs], "y_val": yt[vs], "X_cal": Xt_[vc], "y_cal": yt[vc],
            "X_te": Xt_[te], "y_te": yt[te], "pid_te": pt[te], "X_src_te": Xs_[ss == "test"], "y_src_te": ys[ss == "test"], "pid_src_te": ps[ss == "test"],
            "pool_positions": np.flatnonzero(pool), "te_positions": np.flatnonzero(te)}

def _jsonl_path(tag, kind="sweep"): return f"{RES_DIR}/{kind}_{tag}.jsonl"
def _jsonl_read(path):
    return [json.loads(l) for l in open(path)] if os.path.exists(path) else []
def _jsonl_append(path, row):
    with open(path, "a") as f: f.write(json.dumps(row, default=_jsonable) + "\n")

def evaluate_ft(T, p_t, p_s):
    return {"auroc_tgt": float(roc_auc_score(T["y_te"], p_t)), "auprc_tgt": float(average_precision_score(T["y_te"], p_t)),
            "auroc_src": float(roc_auc_score(T["y_src_te"], p_s)), "auprc_src": float(average_precision_score(T["y_src_te"], p_s))}

def fine_tune_model(key, tag, T, X_sub, y_sub, seed, n_feat=None, states=None, xgb_src=None):
    """Fine-tune / train one model on the subsample; returns (target-test preds, source-test preds)."""
    states = states if states is not None else EXP[tag]["states"]; xgb_src = xgb_src if xgb_src is not None else EXP[tag]["xgb"]
    seed_everything(seed); torch.cuda.empty_cache()
    n_pos = int(y_sub.sum())
    if key in ("MHCA-VAE", "FTT", "DA-VAE", "FTT-DANN"):
        trl, vll = make_loaders(X_sub, y_sub, T["X_val"], T["y_val"])
        if key == "MHCA-VAE":
            m = new_mhcavae(); m.load_state_dict(states["MHCA-VAE"]); m = train_mhcavae(m, trl, vll, CFG["ft_epochs"], lr=1e-4, pat=CFG["ft_patience"]); f = lambda X: pred_mhcavae(m, X)
        elif key == "FTT":
            m = train_ftt_from_state(states["FTT"], trl, vll, CFG["ft_epochs"], lr=1e-4, pat=CFG["ft_patience"], n_feat=n_feat); f = lambda X: pred_ftt(m, X)
        elif key == "DA-VAE":
            m = fine_tune_davae(states["DA-VAE"], X_sub, y_sub, T["X_val"], T["y_val"], CFG["ft_epochs"], patience=CFG["ft_patience"]); f = lambda X: pred_davae(m, X)
        else:
            m = fine_tune_fttdann(states["FTT-DANN"], X_sub, y_sub, T["X_val"], T["y_val"], CFG["ft_epochs"], patience=CFG["ft_patience"]); f = lambda X: pred_dann(m, X)
        out = (f(T["X_te"]), f(T["X_src_te"])); del m; torch.cuda.empty_cache(); gc.collect(); return out
    X_bal, y_bal = balance_cohort(X_sub, y_sub, seed) if n_pos > 10 else (X_sub, y_sub)
    if n_pos < 5: return None
    if key == "XGB":
        m = XGBClassifier(**XGB_FT_PARAMS, random_state=seed).fit(X_bal, y_bal.astype(int))
    elif key == "XGB-warm":
        m = XGBClassifier(**XGB_WARM_PARAMS, random_state=seed).fit(X_bal, y_bal.astype(int), xgb_model=xgb_src.get_booster(),
                                                                       eval_set=[(T["X_val"], T["y_val"].astype(int))], verbose=False)
    else: raise ValueError(key)
    return xgb_predict(m, T["X_te"]), xgb_predict(m, T["X_src_te"])

def zero_shot_rows(tag, T, states=None, xgb_src=None, models=None):
    states = states if states is not None else EXP[tag]["states"]; xgb_src = xgb_src if xgb_src is not None else EXP[tag]["xgb"]
    rows = {}
    for key in models or (["MHCA-VAE", "FTT", "DA-VAE", "XGB"] + (["FTT-DANN"] if "FTT-DANN" in states else [])):
        if key == "MHCA-VAE": m = new_mhcavae(); m.load_state_dict(states[key]); f = lambda X: pred_mhcavae(m, X)
        elif key == "FTT": m = new_ftt(T["X_te"].shape[1]); m.load_state_dict(states[key]); f = lambda X: pred_ftt(m, X)
        elif key == "DA-VAE": m = new_davae(); m.load_state_dict(states[key]); f = lambda X: pred_davae(m, X)
        elif key == "FTT-DANN": m = new_dann(); m.load_state_dict(states[key]); f = lambda X: pred_dann(m, X)
        else: f = lambda X: xgb_predict(xgb_src, X)
        rows[key] = (f(T["X_te"]), f(T["X_src_te"]))
        if key != "XGB": del m
    torch.cuda.empty_cache(); return rows

def run_transfer_sweep(src_name, tgt_name, include_dann):
    tag = dir_tag(src_name, tgt_name); T = prepare_transfer(src_name, tgt_name); path = _jsonl_path(tag)
    models = ["MHCA-VAE", "FTT", "DA-VAE"] + (["FTT-DANN"] if include_dann and "FTT-DANN" in EXP[tag]["states"] else []) + ["XGB", "XGB-warm"]
    done = {(r["n_ft"], r["draw"], r["model"]) for r in _jsonl_read(path)} if RESUME else set()
    if not RESUME and os.path.exists(path): os.remove(path)
    print(f"\n{'='*70}\nTRANSFER SWEEP {tag}: pool {len(T['X_pool']):,} ({T['y_pool'].mean():.1%} ESBL+) | val {len(T['X_val']):,} | target test {len(T['X_te']):,} | source test {len(T['X_src_te']):,}\n{'='*70}")
    # zero-shot rows on the target test split
    if any((0, 0, k) not in done for k in models if k != "XGB-warm"):
        for key, (p_t, p_s) in zero_shot_rows(tag, T).items():
            if (0, 0, key) in done: continue
            save_preds(f"sw_{tag}_{key}_n0_r0", target=p_t, source=p_s)
            _jsonl_append(path, {"n_ft": 0, "draw": 0, "model": key, **evaluate_ft(T, p_t, p_s), "n_pos": int(T["y_pool"].sum()), "seconds": 0.0})
    pool_n = len(T["X_pool"])
    schedule = [(n, r) for n in CFG["budgets"] for r in range(CFG["R_draws"])] + [(pool_n, r) for r in range(CFG["all_seeds"])]
    for n, r in schedule:
        pos = draw_positions(tag, n, r, pool_n); X_sub, y_sub = T["X_pool"][pos], T["y_pool"][pos]; n_pos = int(y_sub.sum())
        todo = [k for k in models if (n, r, k) not in done]
        if not todo: continue
        t0 = time.time(); print(f"  n={n:,} draw {r} ({n_pos} ESBL+): ", end="")
        for key in todo:
            t1 = time.time(); res = fine_tune_model(key, tag, T, X_sub, y_sub, stage_seed("ft", tag, key, n, r))
            if res is None:
                _jsonl_append(path, {"n_ft": n, "draw": r, "model": key, "auroc_tgt": None, "auprc_tgt": None, "auroc_src": None, "auprc_src": None, "n_pos": n_pos, "seconds": 0.0}); continue
            p_t, p_s = res; save_preds(f"sw_{tag}_{key}_n{n}_r{r}", target=p_t, source=p_s)
            row = {"n_ft": n, "draw": r, "model": key, **evaluate_ft(T, p_t, p_s), "n_pos": n_pos, "seconds": time.time() - t1}
            _jsonl_append(path, row); print(f"{key} {row['auroc_tgt']:.4f} ", end="")
        print(f"| {time.time()-t0:.0f}s")
    return T

def run_n500_seed_study(src_name, tgt_name):
    """Initialisation noise at a fixed subsample: n=500, draw 0, several training seeds."""
    tag = dir_tag(src_name, tgt_name); T = prepare_transfer(src_name, tgt_name); path = _jsonl_path(tag, "n500seeds")
    done = {(r["seed"], r["model"]) for r in _jsonl_read(path)} if RESUME else set()
    n = min(500, len(T["X_pool"])); pos = draw_positions(tag, n, 0, len(T["X_pool"])); X_sub, y_sub = T["X_pool"][pos], T["y_pool"][pos]
    for seed in CFG["n500_seeds"]:
        for key in ("FTT", "MHCA-VAE", "XGB"):
            if (seed, key) in done: continue
            res = fine_tune_model(key, tag, T, X_sub, y_sub, stage_seed("n500seed", tag, key, seed))
            if res is None: continue
            p_t, p_s = res
            _jsonl_append(path, {"seed": seed, "model": key, "n_ft": n, **evaluate_ft(T, p_t, p_s)})
            print(f"  n=500 draw 0 seed {seed} {key}: target-test AUROC {roc_auc_score(T['y_te'], p_t):.4f}")

In [16]:
t_sw = time.time()
TRANSFER = {}
TRANSFER["MGB2Stanford"] = run_transfer_sweep("MGB", "Stanford", include_dann=True)
TRANSFER["Stanford2MGB"] = run_transfer_sweep("Stanford", "MGB", include_dann=False)
run_n500_seed_study("MGB", "Stanford")
print(f"\nTransfer sweeps done in {(time.time()-t_sw)/60:.1f} min")


TRANSFER SWEEP MGB2Stanford: pool 48,736 (9.1% ESBL+) | val 6,223 | target test 15,151 | source test 24,063


  n=500 draw 0 (34 ESBL+): 

MHCA-VAE 0.7172 

FTT 0.7372 

DA-VAE 0.7201 

...

  n=500 draw 0 seed 123 MHCA-VAE: target-test AUROC 0.7170


  n=500 draw 0 seed 123 XGB: target-test AUROC 0.6208


  n=500 draw 0 seed 2024 FTT: target-test AUROC 0.7378


  n=500 draw 0 seed 2024 MHCA-VAE: target-test AUROC 0.7191


  n=500 draw 0 seed 2024 XGB: target-test AUROC 0.6155

Transfer sweeps done in 41.4 min


## Results from disk: Table 2, Figures 1–2, transfer tables, Table 3, Figure 3

Everything below reads the saved predictions and JSONL sweep results; nothing is retrained. All confidence
intervals are percentile intervals from a patient-cluster bootstrap (2,000 replicates; the resampling unit is
the patient; the same replicate weights are applied to every model on a given evaluation set, so paired
differences use identical resamples).

In [17]:
t0 = time.time()
def target_weights(site): return boot_weights(SITE_DF[site]["patient_id"].to_numpy(), key=f"target_{site}")

TABLE2 = {}
for src, tgt in DIRS:
    tag = dir_tag(src, tgt); yt = SITE_DF[tgt]["ESBL"].to_numpy(dtype=np.float64); W = target_weights(tgt); TABLE2[tag] = {}
    for key in MODEL_KEYS:
        if key not in EXP[tag]["preds"]: continue
        pr = EXP[tag]["preds"][key]; p, pc = pr["target_raw"].astype(np.float64), pr["target_cal"].astype(np.float64)
        m = eval_with_ci(yt, p, W)
        m.update({"brier_cal": float(brier_score_loss(yt, pc)), "ece_raw": ece(yt, p), "ece_cal": ece(yt, pc), "auroc_cal": float(roc_auc_score(yt, pc))})
        TABLE2[tag][key] = m
        print(f"  {tag:<13s} {MODEL_NAMES[key]:<16s} AUROC {m['auroc']:.4f} [{m['ci_auroc'][0]:.4f}-{m['ci_auroc'][1]:.4f}] AUPRC {m['auprc']:.4f} "
              f"[{m['ci_auprc'][0]:.4f}-{m['ci_auprc'][1]:.4f}] Brier raw {m['brier']:.4f} / isotonic {m['brier_cal']:.4f}")
    aucs = {k: v["auroc"] for k, v in TABLE2[tag].items()}
    rset("table2_summary", tag, value={"auroc_min": min(aucs.values()), "auroc_max": max(aucs.values()), "best_model": max(aucs, key=aucs.get),
                                        "ranking": sorted(aucs, key=aucs.get, reverse=True)})
rset("table2", value=TABLE2)

# ── Table 2 LaTeX body (raw scores for all models; best per column in bold) ──
def _fmt(v, nd=3): return f"{v:.{nd}f}"
lines = []
for key in MODEL_KEYS:
    if not all(key in TABLE2[dir_tag(s, t)] for s, t in DIRS): continue
    cells = [MODEL_NAMES[key]]
    for s, t in DIRS:
        tag = dir_tag(s, t); m = TABLE2[tag][key]; col = TABLE2[tag]
        best_auc = max(v["auroc"] for v in col.values()); best_ap = max(v["auprc"] for v in col.values()); best_br = min(v["brier"] for v in col.values())
        b = lambda x, best: (r"\textbf{" + _fmt(x) + "}") if abs(x - best) < 5e-4 else _fmt(x)
        cells += [b(m["auroc"], best_auc), f"{_fmt(m['ci_auroc'][0])}--{_fmt(m['ci_auroc'][1])}", b(m["auprc"], best_ap), b(m["brier"], best_br)]
    lines.append(" & ".join(cells) + r" \\")
write_tex("table2.tex", "\n".join(lines) + "\n")

# ── Figures 1 and 2: four-panel external validation plots ──
FIG_MODELS = ["LR", "XGB", "FTT", "MHCA-VAE", "DA-VAE"]
for (src, tgt), fname in zip(DIRS, ["fig1a_mgb_to_stanford.png", "fig1b_stanford_to_mgb.png"]):
    tag = dir_tag(src, tgt); yt = SITE_DF[tgt]["ESBL"].to_numpy(dtype=np.float64)
    raw = {k: EXP[tag]["preds"][k]["target_raw"] for k in FIG_MODELS if k in EXP[tag]["preds"]}
    cal = {k: EXP[tag]["preds"][k]["target_cal"] for k in FIG_MODELS if k in EXP[tag]["preds"]}
    four_panel_figure(yt, raw, cal, f"{src} -> {tgt} (external validation, natural prevalence {yt.mean():.1%})", f"{PAPER_FIG_DIR}/{fname}")
    # facts for the captions
    for k, p in raw.items():
        order = np.argsort(-p); top = order[:20]
        rset("figure_facts", tag, k, value={"top20_fp": int((yt[top] == 0).sum()), "top1_is_fp": bool(yt[order[0]] == 0), "max_score": float(p.max()),
                                            "n_bins_lt30": int((reliability_bins(yt, cal[k])["n"] < 30).sum())})
save_results(); print(f"Table 2 and Figures 1-2 done in {time.time()-t0:.0f}s")

  MGB2Stanford  LR               AUROC 0.7176 [0.7092-0.7263] AUPRC 0.3459 [0.3257-0.3663] Brier raw 0.1346 / isotonic 0.0708


  MGB2Stanford  XGBoost          AUROC 0.7266 [0.7185-0.7353] AUPRC 0.3564 [0.3366-0.3757] Brier raw 0.1522 / isotonic 0.0708


  MGB2Stanford  FT-Transformer   AUROC 0.7344 [0.7259-0.7428] AUPRC 0.3554 [0.3350-0.3755] Brier raw 0.1373 / isotonic 0.0708


  MGB2Stanford  MHCA-VAE         AUROC 0.7216 [0.7128-0.7301] AUPRC 0.3404 [0.3209-0.3605] Brier raw 0.1305 / isotonic 0.0715


  MGB2Stanford  DA-VAE           AUROC 0.7100 [0.7006-0.7190] AUPRC 0.3459 [0.3257-0.3665] Brier raw 0.3764 / isotonic 0.0714


...

  Stanford2MGB  DA-VAE           AUROC 0.7479 [0.7417-0.7539] AUPRC 0.4467 [0.4317-0.4607] Brier raw 0.6247 / isotonic 0.0973


  Stanford2MGB  FTT-DANN         AUROC 0.7447 [0.7385-0.7506] AUPRC 0.4506 [0.4355-0.4654] Brier raw 0.1523 / isotonic 0.0983


  Stanford2MGB  TabPFN v2.6      AUROC 0.7411 [0.7345-0.7473] AUPRC 0.4643 [0.4501-0.4785] Brier raw 0.0967 / isotonic 0.0962
  wrote cross_site_outputs/results/table2.tex


  saved figures/fig1a_mgb_to_stanford.png


  saved figures/fig1b_stanford_to_mgb.png
Table 2 and Figures 1-2 done in 65s


In [18]:
t0 = time.time()
SWEEP, SWEEP_SUMMARY, DELTA = {}, {}, {}
COMPARATORS = [("FTT", "XGB"), ("FTT", "XGB-warm"), ("MHCA-VAE", "XGB"), ("MHCA-VAE", "XGB-warm"), ("FTT", "XGB0"), ("MHCA-VAE", "XGB0"), ("FTT", "FTT0")]

for src, tgt in DIRS:
    tag = dir_tag(src, tgt); T = TRANSFER[tag]
    df = pd.DataFrame(_jsonl_read(_jsonl_path(tag))); SWEEP[tag] = df
    pool_n = len(T["X_pool"]); df["budget"] = np.where(df["n_ft"] == pool_n, "ALL", df["n_ft"].astype(str))
    zs_src = df[df["n_ft"] == 0].set_index("model")["auroc_src"].to_dict(); zs_src["XGB-warm"] = zs_src.get("XGB")
    df["forget"] = df["auroc_src"] - df["model"].map(zs_src)
    summ = (df.groupby(["model", "budget"]).agg(auroc_mean=("auroc_tgt", "mean"), auroc_sd=("auroc_tgt", "std"), auroc_min=("auroc_tgt", "min"), auroc_max=("auroc_tgt", "max"),
                                              auprc_mean=("auprc_tgt", "mean"), auprc_sd=("auprc_tgt", "std"), forget_mean=("forget", "mean"), forget_sd=("forget", "std"),
                                              n_draws=("draw", "nunique")).reset_index())
    SWEEP_SUMMARY[tag] = summ
    rset("sweep_summary", tag, value={f"{r.model}|{r.budget}": {k: (None if pd.isna(v) else float(v)) if k not in ("model", "budget") else v for k, v in r._asdict().items() if k != "Index"} for r in summ.itertuples()})
    print(f"\n{tag}: mean AUROC on the target test split (SD across draws)")
    piv = summ.pivot(index="budget", columns="model", values="auroc_mean").reindex(["0"] + [str(b) for b in CFG["budgets"]] + ["ALL"])
    print(piv.round(4).to_string())

    # ── paired deltas with cluster-bootstrap CI of the mean-over-draws delta ──
    W = boot_weights(T["pid_te"], key=f"tgt_test_{tgt}"); y = T["y_te"].astype(np.float64); DELTA[tag] = {}
    p0 = {k: load_preds(f"sw_{tag}_{k}_n0_r0")["target"] for k in ("XGB", "FTT")}
    for n in CFG["budgets"]:
        draws = sorted(df[(df["n_ft"] == n)]["draw"].unique())
        for a, b in COMPARATORS:
            deltas, point = [], []
            for r in draws:
                fa = f"sw_{tag}_{a}_n{n}_r{r}"
                fb = {"XGB0": f"sw_{tag}_XGB_n0_r0", "FTT0": f"sw_{tag}_FTT_n0_r0"}.get(b, f"sw_{tag}_{b}_n{n}_r{r}")
                if not (os.path.exists(pred_path(fa)) and os.path.exists(pred_path(fb))): continue
                pa, pb = load_preds(fa)["target"].astype(np.float64), load_preds(fb)["target"].astype(np.float64)
                deltas.append(paired_delta(y, pa, pb, W)); point.append(roc_auc_score(y, pa) - roc_auc_score(y, pb))
            if not deltas: continue
            mean_over_draws = np.mean(np.stack(deltas), axis=0)
            DELTA[tag][f"{a}-{b}|{n}"] = {"delta_mean": float(np.mean(point)), "delta_sd": float(np.std(point, ddof=1)) if len(point) > 1 else 0.0,
                                          "delta_min": float(np.min(point)), "delta_max": float(np.max(point)), "n_draws": len(point),
                                          "ci": ci(mean_over_draws), "ci_primary_draw": ci(deltas[0])}
    rset("delta", tag, value=DELTA[tag])
    for k, v in DELTA[tag].items(): print(f"  {tag} Δ {k:<18s} mean {v['delta_mean']:+.4f} SD {v['delta_sd']:.4f} CI [{v['ci'][0]:+.4f}, {v['ci'][1]:+.4f}]")

# n=500 initialisation-noise study
n5 = pd.DataFrame(_jsonl_read(_jsonl_path("MGB2Stanford", "n500seeds")))
if len(n5):
    s = n5.groupby("model")["auroc_tgt"].agg(["mean", "std", "min", "max", "count"])
    rset("n500_seed_study", "MGB2Stanford", value={m: {k: float(v) for k, v in row.items()} for m, row in s.to_dict("index").items()})
    print("\nn=500 draw 0, training-seed variability (MGB->Stanford):\n" + s.round(4).to_string())

# ── Table 3 (main text): data-efficiency deltas ──
lines = []
for n in CFG["budgets"]:
    for a, b in [("FTT", "XGB"), ("FTT", "XGB-warm"), ("MHCA-VAE", "XGB"), ("MHCA-VAE", "XGB-warm")]:
        cells = [f"{n:,}", f"{MODEL_NAMES[a]} $-$ {MODEL_NAMES[b]}"]
        for s, t in DIRS:
            v = DELTA[dir_tag(s, t)].get(f"{a}-{b}|{n}")
            cells.append("--" if v is None else f"{s3(v['delta_mean'])} $\\pm$ {v['delta_sd']:.3f} & [{s3(v['ci'][0])}, {s3(v['ci'][1])}]")
        lines.append(" & ".join(cells) + r" \\")
write_tex("table3_delta.tex", "\n".join(lines) + "\n")

# ── Supplementary table: mean +- SD across draws per model and budget ──
lines = []
for s, t in DIRS:
    tag = dir_tag(s, t); summ = SWEEP_SUMMARY[tag]
    models = [m for m in ["MHCA-VAE", "FTT", "DA-VAE", "FTT-DANN", "XGB", "XGB-warm"] if m in set(summ["model"])]
    lines.append(r"\midrule" + "\n" + r"\multicolumn{" + str(len(models) + 1) + r"}{l}{\textit{" + f"{s} to {t}" + r"}} \\")
    lines.append("Budget & " + " & ".join(MODEL_NAMES[m] for m in models) + r" \\")
    for budget in ["0"] + [str(b) for b in CFG["budgets"]] + ["ALL"]:
        cells = ["zero-shot" if budget == "0" else budget]
        for m in models:
            r = summ[(summ["model"] == m) & (summ["budget"] == budget)]
            if len(r) == 0 or pd.isna(r["auroc_mean"].iloc[0]): cells.append("--")
            elif budget == "0" or pd.isna(r["auroc_sd"].iloc[0]): cells.append(f"{r['auroc_mean'].iloc[0]:.3f}")
            else: cells.append(f"{r['auroc_mean'].iloc[0]:.3f} $\\pm$ {r['auroc_sd'].iloc[0]:.3f}")
        lines.append(" & ".join(cells) + r" \\")
write_tex("table_transfer.tex", "\n".join(lines) + "\n")

# ── Figure 3 ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for row, (s, t) in enumerate(DIRS):
    tag = dir_tag(s, t); summ = SWEEP_SUMMARY[tag]; pool_n = len(TRANSFER[tag]["X_pool"])
    budgets = ["0"] + [str(b) for b in CFG["budgets"]] + ["ALL"]; xs = np.array([0] + CFG["budgets"] + [pool_n])
    for col, (metric, ylabel) in enumerate([("auroc", "Target AUROC"), ("auprc", "Target AUPRC"), ("forget", "Source AUROC change")]):
        ax = axes[row, col]
        for m in ["MHCA-VAE", "FTT", "DA-VAE", "FTT-DANN", "XGB", "XGB-warm"]:
            sub = summ[summ["model"] == m].set_index("budget").reindex(budgets)
            if sub[f"{metric}_mean"].isna().all(): continue
            mean = sub[f"{metric}_mean"].to_numpy(dtype=float); sd = sub[f"{metric}_sd"].fillna(0).to_numpy(dtype=float)
            c, _ = MODEL_STYLE[m]; ax.plot(xs, mean, f"{MARKERS[m]}-", color=c, lw=2, ms=6, label=MODEL_NAMES[m]); ax.fill_between(xs, mean - sd, mean + sd, color=c, alpha=0.15)
        if row == 0 and metric != "forget" and TABPFN_SWEEP.get(tag):
            tp = pd.DataFrame(TABPFN_SWEEP[tag]); ax.plot(tp["n_ft"], tp[f"{metric}_tgt"], f"{MARKERS['TabPFN']}-", color=MODEL_STYLE["TabPFN"][0], lw=2, ms=9, label="TabPFN v2.6 (context)")
        if metric == "forget": ax.axhline(0, color="black", lw=0.8, ls="--")
        ax.set_xscale("symlog", linthresh=100); ax.set_xlim(-40, pool_n * 1.6); ax.set_xlabel(f"# {t} fine-tuning cultures"); ax.set_ylabel(ylabel)
        ax.set_title(f"{s} -> {t}: {ylabel}", fontweight="bold"); ax.grid(alpha=0.2); ax.legend(fontsize=8)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.suptitle("Transfer data efficiency and source-site forgetting (mean $\\pm$ SD over random target subsamples)", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(f"{PAPER_FIG_DIR}/multimodel_transfer.png", dpi=200, bbox_inches="tight"); plt.close(fig)
save_results(); print(f"Transfer tables, Table 3 and Figure 3 done in {time.time()-t0:.0f}s")


MGB2Stanford: mean AUROC on the target test split (SD across draws)
model   DA-VAE     FTT  FTT-DANN  MHCA-VAE     XGB  XGB-warm
budget                                                      
0       0.7097  0.7303    0.7401    0.7124  0.7248       NaN
500     0.7230  0.7363    0.7416    0.7240  0.6232    0.7299
1000    0.7254  0.7373    0.7418    0.7254  0.6685    0.7307
2000    0.7291  0.7421    0.7456    0.7329  0.6898    0.7362
5000    0.7345  0.7468    0.7495    0.7413  0.7059    0.7413
ALL     0.7545  0.7627    0.7636    0.7582  0.7490    0.7580


  MGB2Stanford Δ FTT-XGB|500        mean +0.1131 SD 0.0217 CI [+0.0998, +0.1258]
  MGB2Stanford Δ FTT-XGB-warm|500   mean +0.0063 SD 0.0081 CI [-0.0038, +0.0150]
  MGB2Stanford Δ MHCA-VAE-XGB|500   mean +0.1008 SD 0.0225 CI [+0.0863, +0.1140]
  MGB2Stanford Δ MHCA-VAE-XGB-warm|500 mean -0.0059 SD 0.0047 CI [-0.0171, +0.0043]
  MGB2Stanford Δ FTT-XGB0|500       mean +0.0115 SD 0.0059 CI [+0.0011, +0.0210]
  MGB2Stanford Δ MHCA-VAE-XGB0|500  mean -0.0008 SD 0.0035 CI [-0.0126, +0.0097]
  MGB2Stanford Δ FTT-FTT0|500       mean +0.0060 SD 0.0059 CI [+0.0026, +0.0094]
  MGB2Stanford Δ FTT-XGB|1000       mean +0.0688 SD 0.0124 CI [+0.0555, +0.0806]
  MGB2Stanford Δ FTT-XGB-warm|1000  mean +0.0066 SD 0.0063 CI [-0.0042, +0.0158]
  MGB2Stanford Δ MHCA-VAE-XGB|1000  mean +0.0569 SD 0.0112 CI [+0.0423, +0.0695]
  MGB2Stanford Δ MHCA-VAE-XGB-warm|1000 mean -0.0053 SD 0.0047 CI [-0.0170, +0.0044]
  MGB2Stanford Δ FTT-XGB0|1000      mean +0.0126 SD 0.0053 CI [+0.0011, +0.0226]
  MGB2Stanford Δ MHCA

  Stanford2MGB Δ FTT-XGB|500        mean +0.0536 SD 0.0167 CI [+0.0464, +0.0604]
  Stanford2MGB Δ FTT-XGB-warm|500   mean +0.0059 SD 0.0030 CI [+0.0018, +0.0099]
  Stanford2MGB Δ MHCA-VAE-XGB|500   mean +0.0539 SD 0.0175 CI [+0.0475, +0.0602]
  Stanford2MGB Δ MHCA-VAE-XGB-warm|500 mean +0.0062 SD 0.0022 CI [+0.0020, +0.0109]
  Stanford2MGB Δ FTT-XGB0|500       mean +0.0087 SD 0.0027 CI [+0.0046, +0.0128]
  Stanford2MGB Δ MHCA-VAE-XGB0|500  mean +0.0090 SD 0.0023 CI [+0.0046, +0.0138]
  Stanford2MGB Δ FTT-FTT0|500       mean +0.0021 SD 0.0027 CI [+0.0013, +0.0029]
  Stanford2MGB Δ FTT-XGB|1000       mean +0.0334 SD 0.0092 CI [+0.0277, +0.0388]
  Stanford2MGB Δ FTT-XGB-warm|1000  mean +0.0076 SD 0.0061 CI [+0.0038, +0.0116]
  Stanford2MGB Δ MHCA-VAE-XGB|1000  mean +0.0335 SD 0.0093 CI [+0.0275, +0.0391]
  Stanford2MGB Δ MHCA-VAE-XGB-warm|1000 mean +0.0077 SD 0.0050 CI [+0.0036, +0.0120]
  Stanford2MGB Δ FTT-XGB0|1000      mean +0.0134 SD 0.0051 CI [+0.0090, +0.0183]
  Stanford2MGB Δ MHCA

Transfer tables, Table 3 and Figure 3 done in 550s


In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# Permutation importance of the source-trained FTT on the held-out source TEST split
# (never used for training, early stopping, calibration or target evaluation)
# ══════════════════════════════════════════════════════════════════════════════
t0 = time.time()
GROUP_COLORS = {"resistance": "#e74c3c", "antibiotics": "#3498db", "demographics": "#9b59b6", "culture_organism": "#2ecc71",
                "temporal_interactions": "#f39c12", "comorbidities": "#1abc9c", "ward": "#95a5a6", "procedures": "#e67e22"}
IMPORTANCE, TOP10 = {}, {}
for (src, tgt), fname in zip(DIRS, ["ftt_importance_m2s.png", "ftt_importance_s2m.png"]):
    tag = dir_tag(src, tgt); path = f"{RES_DIR}/importance_{tag}.csv"
    if RESUME and os.path.exists(path):
        fi = pd.read_csv(path)
    else:
        Xs, ys, ps, ss = site_arrays(SITE_DF[src]); X_te = EXP[tag]["scaler"].transform(Xs[ss == "test"]).astype(np.float32); y_te = ys[ss == "test"]
        m = new_ftt(); m.load_state_dict(EXP[tag]["states"]["FTT"]); m.eval()
        base = roc_auc_score(y_te, pred_ftt(m, X_te)); rng = np.random.RandomState(stage_seed("perm", tag))
        drops = []
        for j, f in enumerate(FEATURES):
            d = []
            for _ in range(CFG["perm_repeats"]):
                Xp = X_te.copy(); Xp[:, j] = rng.permutation(Xp[:, j]); d.append(base - roc_auc_score(y_te, pred_ftt(m, Xp)))
            drops.append(float(np.mean(d)))
        fi = pd.DataFrame({"feature": FEATURES, "importance": drops}).sort_values("importance", ascending=False).reset_index(drop=True)
        fi["rank"] = fi.index + 1; fi["group"] = fi["feature"].map(FEAT_TO_GROUP); fi["baseline_auroc"] = base
        fi.to_csv(path, index=False); del m; torch.cuda.empty_cache()
    IMPORTANCE[tag] = fi; TOP10[tag] = fi.head(10)["feature"].tolist()
    rset("importance", tag, value={"baseline_source_test_auroc": float(fi["baseline_auroc"].iloc[0]), "top10": TOP10[tag],
                                   "values": {r.feature: float(r.importance) for r in fi.itertuples()}})
    print(f"{tag}: source-test baseline AUROC {fi['baseline_auroc'].iloc[0]:.4f}; top 10: {TOP10[tag]}")
    top = fi.head(10)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(10), top["importance"].values[::-1], color=[GROUP_COLORS.get(g, "#bdc3c7") for g in top["group"]][::-1], alpha=0.9, edgecolor="white")
    ax.set_yticks(range(10)); ax.set_yticklabels(top["feature"].values[::-1], fontsize=11)
    ax.set_xlabel(f"AUROC drop on the {src} held-out test split (mean over {CFG['perm_repeats']} permutations)", fontsize=11)
    ax.set_title(f"FT-Transformer permutation importance: {src} -> {tgt} model\nTop 10 features", fontweight="bold", fontsize=13)
    ax.legend(handles=[Patch(color=GROUP_COLORS.get(g, "#bdc3c7"), label=g) for g in top["group"].unique()], fontsize=9, loc="lower right")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False); plt.tight_layout()
    plt.savefig(f"{PAPER_FIG_DIR}/{fname}", dpi=300, bbox_inches="tight", facecolor="white"); plt.close(fig)

STABLE = [f for f in TOP10["MGB2Stanford"] if f in TOP10["Stanford2MGB"]]
rset("importance", "stable_features", value=STABLE)
print(f"Features in both top-10 lists ({len(STABLE)}): {STABLE} | {time.time()-t0:.0f}s")

MGB2Stanford: source-test baseline AUROC 0.7879; top 10: ['prior_ESBL', 'prior_esbl_ast', 'num_prior_orgs', 'org_ecoli', 'T_ceph3_90d', 'prior_n_resistant_abx', 'age_encoded', 'adi_missing', 'org_klebsiella', 'hosp_ward_IP']


Stanford2MGB: source-test baseline AUROC 0.7627; top 10: ['prior_ESBL', 'num_prior_orgs', 'org_ecoli', 'gender_male', 'prior_esbl_ast', 'T_sulfa_90d', 'T_fq_90d', 'days_since_prior_org', 'age_encoded', 'any_proc_30d']


Features in both top-10 lists (5): ['prior_ESBL', 'prior_esbl_ast', 'num_prior_orgs', 'org_ecoli', 'age_encoded'] | 48s


In [20]:
# ══════════════════════════════════════════════════════════════════════════════
# Reduced-feature FT-Transformer models (top-10 per direction, and the cross-direction stable set),
# trained and fine-tuned on exactly the splits and draw-0 subsamples of the main sweep
# ══════════════════════════════════════════════════════════════════════════════
t0 = time.time()
def run_reduced_feature_sweep(feature_list, name, src, tgt):
    tag = dir_tag(src, tgt); rtag = f"{tag}_{name}"; idx = [FEATURES.index(f) for f in feature_list]; nf = len(idx)
    Xs, ys, ps, ss = site_arrays(SITE_DF[src]); Xt, yt, pt, st = site_arrays(SITE_DF[tgt])
    tr, vs = (ss == "train"), (ss == "val_sel")
    sc = StandardScaler().fit(Xs[tr][:, idx]); Xs_ = sc.transform(Xs[:, idx]).astype(np.float32); Xt_ = sc.transform(Xt[:, idx]).astype(np.float32)
    if have_ckpt(f"{rtag}_FTT.pt") and have_preds(f"zs_{rtag}_FTT"):
        state = load_state(f"{rtag}_FTT.pt"); p_full = load_preds(f"zs_{rtag}_FTT")["target_raw"]
    else:
        seed_everything(stage_seed("reduced", rtag)); trl, vll = make_loaders(Xs_[tr], ys[tr], Xs_[vs], ys[vs])
        m = train_ftt(new_ftt(nf), trl, vll, CFG["epochs"], lr=5e-4, pat=CFG["patience"]); state = state_of(m); save_state(f"{rtag}_FTT.pt", state)
        p_full = pred_ftt(m, Xt_); save_preds(f"zs_{rtag}_FTT", target_raw=p_full); del m; torch.cuda.empty_cache()
    vs_t = st == "val_sel"
    T = {"tag": tag, "X_pool": Xt_[st == "train"], "y_pool": yt[st == "train"], "X_val": Xt_[vs_t], "y_val": yt[vs_t],
         "X_te": Xt_[st == "test"], "y_te": yt[st == "test"], "pid_te": pt[st == "test"], "X_src_te": Xs_[ss == "test"], "y_src_te": ys[ss == "test"]}
    path = _jsonl_path(rtag, "reduced"); done = {(r["n_ft"], r["draw"]) for r in _jsonl_read(path)} if RESUME else set()
    if not RESUME and os.path.exists(path): os.remove(path)
    if (0, 0) not in done:
        p_t, p_s = zero_shot_rows(tag, T, states={"FTT": state}, models=["FTT"])["FTT"]
        save_preds(f"rd_{rtag}_n0_r0", target=p_t, source=p_s); _jsonl_append(path, {"n_ft": 0, "draw": 0, "model": "FTT", **evaluate_ft(T, p_t, p_s)})
    pool_n = len(T["X_pool"])
    for n in CFG["budgets"] + [pool_n]:
        if (n, 0) in done: continue
        pos = draw_positions(tag, n, 0, pool_n)
        res = fine_tune_model("FTT", tag, T, T["X_pool"][pos], T["y_pool"][pos], stage_seed("ft", rtag, "FTT", n, 0), n_feat=nf, states={"FTT": state})
        p_t, p_s = res; save_preds(f"rd_{rtag}_n{n}_r0", target=p_t, source=p_s)
        _jsonl_append(path, {"n_ft": n, "draw": 0, "model": "FTT", **evaluate_ft(T, p_t, p_s)})
    rows = pd.DataFrame(_jsonl_read(path)).sort_values("n_ft")
    # zero-shot on the FULL target site with a paired cluster-bootstrap delta vs the 46-feature FTT
    W = target_weights(tgt); ytf = yt.astype(np.float64); p46 = EXP[tag]["preds"]["FTT"]["target_raw"].astype(np.float64)
    full = eval_with_ci(ytf, p_full.astype(np.float64), W, which=("auroc", "auprc"))
    d = paired_delta(ytf, p_full.astype(np.float64), p46, W)
    out = {"features": feature_list, "n_features": nf, "zero_shot_full_target": full, "delta_vs_46_full_target": {"delta": full["auroc"] - float(roc_auc_score(ytf, p46)), "ci": ci(d)},
           "sweep_target_test": rows[["n_ft", "auroc_tgt", "auprc_tgt", "auroc_src"]].to_dict("records"), "pool_n": int(pool_n)}
    rset("reduced", rtag, value=out)
    print(f"  {rtag} ({nf} features): zero-shot full-target AUROC {full['auroc']:.4f} [{full['ci_auroc'][0]:.4f}-{full['ci_auroc'][1]:.4f}] "
          f"(Δ vs 46-feature {out['delta_vs_46_full_target']['delta']:+.4f} CI [{out['delta_vs_46_full_target']['ci'][0]:+.4f}, {out['delta_vs_46_full_target']['ci'][1]:+.4f}]); "
          f"ALL {rows['auroc_tgt'].iloc[-1]:.4f} | {time.time()-t0:.0f}s")
    return out

REDUCED = {}
REDUCED["MGB2Stanford_top10"] = run_reduced_feature_sweep(TOP10["MGB2Stanford"], "top10", "MGB", "Stanford")
REDUCED["Stanford2MGB_top10"] = run_reduced_feature_sweep(TOP10["Stanford2MGB"], "top10", "Stanford", "MGB")
if len(STABLE) >= 2:
    REDUCED["MGB2Stanford_stable"] = run_reduced_feature_sweep(STABLE, "stable", "MGB", "Stanford")
    REDUCED["Stanford2MGB_stable"] = run_reduced_feature_sweep(STABLE, "stable", "Stanford", "MGB")

# ── Supplementary Figure 4: 46-feature vs top-10 vs stable set, target test split, draw 0 ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for row, (s, t) in enumerate(DIRS):
    tag = dir_tag(s, t); sw = SWEEP[tag]; pool_n = len(TRANSFER[tag]["X_pool"])
    full46 = sw[(sw["model"] == "FTT") & (sw["draw"] == 0)].sort_values("n_ft")
    series = {"46 features (draw 0)": full46, f"Top-10 ({s}-selected)": pd.DataFrame(REDUCED[f"{tag}_top10"]["sweep_target_test"])}
    if f"{tag}_stable" in REDUCED: series[f"Stable {len(STABLE)}-feature set"] = pd.DataFrame(REDUCED[f"{tag}_stable"]["sweep_target_test"])
    for col, metric in enumerate(["auroc_tgt", "auprc_tgt"]):
        ax = axes[row, col]
        for (lab, d), c, mk in zip(series.items(), ["#3498db", "#e67e22", "#2ecc71"], ["s", "o", "D"]):
            d = d.sort_values("n_ft"); xs = np.arange(len(d)); ax.plot(xs, d[metric], f"{mk}-", color=c, lw=2, ms=7, label=lab)
            for x, v in zip(xs, d[metric]): ax.annotate(f"{v:.3f}", (x, v), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=7, color=c)
            ax.set_xticks(xs); ax.set_xticklabels(["0 (zs)"] + [f"{int(n):,}" if n < pool_n else "ALL" for n in d["n_ft"].iloc[1:]])
        ax.set_xlabel(f"{t} fine-tuning cultures"); ax.set_ylabel("AUROC" if metric == "auroc_tgt" else "AUPRC"); ax.margins(y=0.25)
        ax.set_title(f"{s} -> {t}: {'AUROC' if metric == 'auroc_tgt' else 'AUPRC'} on the target test split", fontweight="bold"); ax.grid(alpha=0.15); ax.legend(fontsize=9)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.savefig(f"{PAPER_FIG_DIR}/suppfig4_reduced_features.png", dpi=300, bbox_inches="tight", facecolor="white"); plt.close(fig)

lines = []
for k, v in REDUCED.items():
    zs = v["zero_shot_full_target"]; dl = v["delta_vs_46_full_target"]; sw_ = pd.DataFrame(v["sweep_target_test"]).sort_values("n_ft")
    lines.append(f"{k.replace('_', ' ').replace('2', ' to ')} & {v['n_features']} & {zs['auroc']:.3f} [{zs['ci_auroc'][0]:.3f}--{zs['ci_auroc'][1]:.3f}] & "
                 f"{s3(dl['delta'])} [{s3(dl['ci'][0])}, {s3(dl['ci'][1])}] & {sw_['auroc_tgt'].iloc[-1]:.3f} \\\\")
write_tex("table_reduced.tex", "\n".join(lines) + "\n"); save_results()

  MGB2Stanford_top10 (10 features): zero-shot full-target AUROC 0.7034 [0.6947-0.7120] (Δ vs 46-feature -0.0310 CI [-0.0352, -0.0268]); ALL 0.7205 | 192s


  Stanford2MGB_top10 (10 features): zero-shot full-target AUROC 0.7352 [0.7289-0.7413] (Δ vs 46-feature -0.0165 CI [-0.0195, -0.0133]); ALL 0.7524 | 381s


  MGB2Stanford_stable (5 features): zero-shot full-target AUROC 0.6976 [0.6886-0.7065] (Δ vs 46-feature -0.0368 CI [-0.0437, -0.0299]); ALL 0.7106 | 509s


  Stanford2MGB_stable (5 features): zero-shot full-target AUROC 0.7122 [0.7059-0.7182] (Δ vs 46-feature -0.0394 CI [-0.0440, -0.0347]); ALL 0.7251 | 720s


  wrote cross_site_outputs/results/table_reduced.tex


In [21]:
# ══════════════════════════════════════════════════════════════════════════════
# Subgroup analysis (FTT, 46 features, zero-shot, full target site) with cluster-bootstrap CIs
# ══════════════════════════════════════════════════════════════════════════════
t0 = time.time()
AGE_BINS = {"18-34": [1, 2], "35-54": [3, 4], "55-64": [5], "65-74": [6], ">=75": [7, 8, 9]}
def subgroup_masks(df):
    m = {}
    for lab, codes in AGE_BINS.items(): m[f"Age {lab}"] = df["age_encoded"].isin(codes)
    m["Male"] = df["gender_male"] == 1; m["Female"] = df["gender_male"] == 0
    ok = df["adi_missing"] == 0; adi = df["adi_score_clean"]   # half-open thirds: MGB scores are non-integer
    m["ADI low (0-33)"] = ok & (adi < 34); m["ADI mid (34-66)"] = ok & (adi >= 34) & (adi < 67)
    m["ADI high (67-100)"] = ok & (adi >= 67); m["ADI missing"] = df["adi_missing"] == 1
    m["Prior ESBL+"] = df["prior_ESBL"] == 1; m["No prior ESBL"] = df["prior_ESBL"] == 0
    return m
SUBGROUP_ORDER = list(subgroup_masks(mgb).keys())
SUBGROUPS = {}
for src, tgt in DIRS:
    tag = dir_tag(src, tgt); df = SITE_DF[tgt]; y = df["ESBL"].to_numpy(dtype=np.float64); p = EXP[tag]["preds"]["FTT"]["target_raw"].astype(np.float64); pid = df["patient_id"].to_numpy()
    SUBGROUPS[tag] = {}
    for lab, mask in subgroup_masks(df).items():
        mk = mask.to_numpy(); ys, ps_, pids = y[mk], p[mk], pid[mk]
        if ys.sum() < 5 or (1 - ys).sum() < 5: continue
        W = boot_weights(pids, key=f"sub_{tgt}_{lab}")
        m = eval_with_ci(ys, ps_, W, which=("auroc", "auprc")); m.update({"n": int(mk.sum()), "n_pos": int(ys.sum()), "prevalence": float(ys.mean())})
        SUBGROUPS[tag][lab] = m
    print(f"{tag}: " + "; ".join(f"{k} {v['auroc']:.3f} [{v['ci_auroc'][0]:.3f}-{v['ci_auroc'][1]:.3f}]" for k, v in SUBGROUPS[tag].items()))
rset("subgroups", value=SUBGROUPS)

# Supplementary Table S3 body
lines = []
sections = [("Age group", [k for k in SUBGROUP_ORDER if k.startswith("Age")]), ("Sex", ["Male", "Female"]),
            ("Area Deprivation Index", [k for k in SUBGROUP_ORDER if k.startswith("ADI")]), ("Prior ESBL history", ["Prior ESBL+", "No prior ESBL"])]
def _sg(tag, lab):
    v = SUBGROUPS[tag].get(lab); return "-- & -- & --" if v is None else f"{v['n']:,} & {v['auroc']:.3f} [{v['ci_auroc'][0]:.3f}--{v['ci_auroc'][1]:.3f}] & {v['auprc']:.3f}"
ov = {dir_tag(s, t): TABLE2[dir_tag(s, t)]["FTT"] for s, t in DIRS}
lines.append(r"\textit{Overall} & " + " & ".join(f"{len(SITE_DF[t]):,} & {ov[dir_tag(s,t)]['auroc']:.3f} [{ov[dir_tag(s,t)]['ci_auroc'][0]:.3f}--{ov[dir_tag(s,t)]['ci_auroc'][1]:.3f}] & {ov[dir_tag(s,t)]['auprc']:.3f}" for s, t in DIRS) + r" \\")
for sec, labs in sections:
    lines.append(r"\midrule" + "\n" + r"\textit{" + sec + r"} & & & & & & \\")
    for lab in labs:
        lab_tex = lab.replace(">=", r"$\geq$")
        lines.append(f"\\quad {lab_tex} & " + " & ".join(_sg(dir_tag(s, t), lab) for s, t in DIRS) + r" \\")
write_tex("table_subgroup.tex", "\n".join(lines) + "\n")

# Supplementary Figure 3 with 95% CI error bars
fig, ax = plt.subplots(figsize=(16, 7)); x = np.arange(len(SUBGROUP_ORDER)); w = 0.38
for off, (s, t), c in zip([-w / 2, w / 2], DIRS, ["#3498db", "#e74c3c"]):
    tag = dir_tag(s, t); vals = np.array([SUBGROUPS[tag].get(k, {}).get("auroc", np.nan) for k in SUBGROUP_ORDER])
    lo = np.array([SUBGROUPS[tag].get(k, {}).get("ci_auroc", [np.nan, np.nan])[0] for k in SUBGROUP_ORDER]); hi = np.array([SUBGROUPS[tag].get(k, {}).get("ci_auroc", [np.nan, np.nan])[1] for k in SUBGROUP_ORDER])
    ok = ~np.isnan(vals)
    ax.bar(x[ok] + off, vals[ok], w, color=c, alpha=0.85, label=f"{s} -> {t}", yerr=[vals[ok] - lo[ok], hi[ok] - vals[ok]], capsize=3, error_kw={"lw": 1})
    ax.axhline(ov[tag]["auroc"], color=c, lw=1.4, ls="--", alpha=0.7, label=f"{s} -> {t} overall ({ov[tag]['auroc']:.3f})")
ax.set_xticks(x); ax.set_xticklabels(SUBGROUP_ORDER, rotation=35, ha="right", fontsize=10); ax.set_ylabel("AUROC", fontsize=12); ax.set_ylim(0.5, 0.9)
ax.set_title("Subgroup AUROC of the zero-shot FT-Transformer (46 features) with patient-cluster bootstrap 95% CIs", fontweight="bold", fontsize=13)
ax.legend(fontsize=10, loc="upper right"); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
for xpos in [4.5, 6.5, 10.5]: ax.axvline(xpos, color="gray", lw=0.8, ls=":", alpha=0.5)
plt.tight_layout(); plt.savefig(f"{PAPER_FIG_DIR}/suppfig3_subgroups.png", dpi=300, bbox_inches="tight", facecolor="white"); plt.close(fig)
save_results(); print(f"Subgroups done in {time.time()-t0:.0f}s")

MGB2Stanford: Age 18-34 0.691 [0.660-0.718]; Age 35-54 0.709 [0.688-0.729]; Age 55-64 0.739 [0.720-0.757]; Age 65-74 0.736 [0.719-0.753]; Age >=75 0.742 [0.727-0.757]; Male 0.734 [0.717-0.749]; Female 0.716 [0.706-0.726]; ADI low (0-33) 0.741 [0.731-0.750]; ADI mid (34-66) 0.715 [0.680-0.747]; ADI high (67-100) 0.809 [0.730-0.867]; ADI missing 0.706 [0.686-0.724]; Prior ESBL+ 0.642 [0.616-0.667]; No prior ESBL 0.644 [0.635-0.652]
Stanford2MGB: Age 18-34 0.708 [0.685-0.729]; Age 35-54 0.743 [0.727-0.759]; Age 55-64 0.759 [0.743-0.773]; Age 65-74 0.754 [0.741-0.766]; Age >=75 0.738 [0.729-0.747]; Male 0.768 [0.757-0.779]; Female 0.728 [0.721-0.735]; ADI low (0-33) 0.782 [0.775-0.790]; ADI mid (34-66) 0.804 [0.779-0.826]; ADI high (67-100) 0.745 [0.562-0.863]; ADI missing 0.719 [0.710-0.728]; Prior ESBL+ 0.664 [0.647-0.681]; No prior ESBL 0.639 [0.632-0.645]
  wrote cross_site_outputs_v14/results/table_subgroup.tex
Subgroups done in 47s


In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# Clinical operating point: threshold chosen at 90% sensitivity on the SOURCE calibration split (raw FTT scores),
# then applied unchanged to the target site; a locally re-thresholded variant uses the target fine-tuning
# validation split and is evaluated on the target test split.
# ══════════════════════════════════════════════════════════════════════════════
t0 = time.time()
def threshold_at_sensitivity(y, p, target=0.90):
    fpr, tpr, thr = roc_curve(y, p); k = int(np.argmax(tpr >= target)); return float(thr[k])
def op_metrics(y, p, thr, W):
    pt = weighted_confusion(y, p, thr, np.ones(len(y), dtype=np.uint8)); boots = {k: [] for k in ("sens", "spec", "ppv", "npv")}
    for w in W:
        r = weighted_confusion(y, p, thr, w)
        for k in boots: boots[k].append(r[k])
    return {k: pt[k] for k in ("sens", "spec", "ppv", "npv", "tp", "fp", "fn", "tn")} | {f"ci_{k}": ci(v) for k, v in boots.items()}
OPERATING = {}
for src, tgt in DIRS:
    tag = dir_tag(src, tgt); df = SITE_DF[tgt]; y = df["ESBL"].to_numpy(dtype=np.float64); st = df["split"].to_numpy()
    pr = EXP[tag]["preds"]["FTT"]; p = pr["target_raw"].astype(np.float64)
    ys = SITE_DF[src]["ESBL"].to_numpy(dtype=np.float64); ss = SITE_DF[src]["split"].to_numpy()
    thr = threshold_at_sensitivity(ys[ss == "val_cal"], pr["valcal_raw"].astype(np.float64))
    W = target_weights(tgt); res = op_metrics(y, p, thr, W); prev = float(y.mean())
    res.update({"threshold": thr, "prevalence": prev, "npv_baseline": 1 - prev, "n": int(len(y)), "source": "source val_cal split, 90% sensitivity"})
    # local re-threshold on the target fine-tuning validation split, evaluated on the target test split
    thr_loc = threshold_at_sensitivity(y[st == "val_sel"], p[st == "val_sel"])
    te = st == "test"; W_te = boot_weights(df["patient_id"].to_numpy()[te], key=f"tgt_test_{tgt}")
    loc = op_metrics(y[te], p[te], thr_loc, W_te); loc.update({"threshold": thr_loc, "prevalence": float(y[te].mean()), "npv_baseline": 1 - float(y[te].mean()), "n": int(te.sum())})
    # state transitions at the source-derived threshold
    trans = {}
    for lab, mask in [("no_prior_esbl", df["prior_ESBL"].to_numpy() == 0), ("prior_esbl_pos", df["prior_ESBL"].to_numpy() == 1)]:
        ym, pm = y[mask], p[mask]; Wm = boot_weights(df["patient_id"].to_numpy()[mask], key=f"sub_{tgt}_{'No prior ESBL' if lab == 'no_prior_esbl' else 'Prior ESBL+'}")
        r = op_metrics(ym, pm, thr, Wm); r.update({"n": int(mask.sum()), "prevalence": float(ym.mean()), "auroc": float(roc_auc_score(ym, pm)), "ci_auroc": ci(boot_metrics(ym, pm, Wm, ("auroc",))["auroc"])})
        trans[lab] = r
    OPERATING[tag] = {"source_threshold": res, "local_threshold": loc, "transitions": trans}
    print(f"{tag}: thr {thr:.3f} -> sens {res['sens']:.3f} [{res['ci_sens'][0]:.3f}-{res['ci_sens'][1]:.3f}] spec {res['spec']:.3f} PPV {res['ppv']:.3f} NPV {res['npv']:.3f} [{res['ci_npv'][0]:.3f}-{res['ci_npv'][1]:.3f}] (baseline NPV {1-prev:.3f}); "
          f"local thr {thr_loc:.3f} on test: sens {loc['sens']:.3f} NPV {loc['npv']:.3f}; no-prior-ESBL NPV {trans['no_prior_esbl']['npv']:.3f} sens {trans['no_prior_esbl']['sens']:.3f}; prior-ESBL+ spec {trans['prior_esbl_pos']['spec']:.3f}")
rset("operating_point", value=OPERATING)
lines = []
for s, t in DIRS:
    r = OPERATING[dir_tag(s, t)]["source_threshold"]; l = OPERATING[dir_tag(s, t)]["local_threshold"]
    pc = lambda v, c: f"{v:.1%} [{c[0]:.1%}--{c[1]:.1%}]".replace("%", r"\%")
    lines.append(f"{s} to {t} (prevalence {r['prevalence']:.1%}) & source & {r['threshold']:.3f} & {pc(r['sens'], r['ci_sens'])} & {pc(r['spec'], r['ci_spec'])} & {pc(r['ppv'], r['ci_ppv'])} & {pc(r['npv'], r['ci_npv'])} & {r['npv_baseline']:.1%} \\\\".replace("%)", r"\%)").replace("& " + f"{r['npv_baseline']:.1%}", "& " + f"{r['npv_baseline']:.1%}".replace("%", r"\%")))
    lines.append(f" & target (test split) & {l['threshold']:.3f} & {pc(l['sens'], l['ci_sens'])} & {pc(l['spec'], l['ci_spec'])} & {pc(l['ppv'], l['ci_ppv'])} & {pc(l['npv'], l['ci_npv'])} & {l['npv_baseline']:.1%} \\\\".replace("& " + f"{l['npv_baseline']:.1%}", "& " + f"{l['npv_baseline']:.1%}".replace("%", r"\%")))
write_tex("table_clinical.tex", "\n".join(lines) + "\n"); save_results(); print(f"Operating point done in {time.time()-t0:.0f}s")

MGB2Stanford: thr 0.321 -> sens 0.821 [0.811-0.831] spec 0.420 PPV 0.125 NPV 0.959 [0.956-0.961] (baseline NPV 0.908); local thr 0.304 on test: sens 0.902 NPV 0.965; no-prior-ESBL NPV 0.959 sens 0.750; prior-ESBL+ spec 0.019


Stanford2MGB: thr 0.306 -> sens 0.908 [0.903-0.913] spec 0.261 PPV 0.165 NPV 0.947 [0.944-0.949] (baseline NPV 0.862); local thr 0.315 on test: sens 0.883 NPV 0.943; no-prior-ESBL NPV 0.947 sens 0.856; prior-ESBL+ spec 0.000
  wrote cross_site_outputs/results/table_clinical.tex
Operating point done in 7s


In [23]:
# ══════════════════════════════════════════════════════════════════════════════
# Sensitivity analyses (evaluation only, zero-shot FTT and XGBoost on the target site)
#  (a) first culture per patient  (b) index culture per 30-day episode
#  (c) cultures with >= 1 CLSI screening agent tested  (d) MGB cultures with a confirmatory ESBL result (label = confirmatory result)
# ══════════════════════════════════════════════════════════════════════════════
t0 = time.time(); SENS = {}
for src, tgt in DIRS:
    tag = dir_tag(src, tgt); df = SITE_DF[tgt]; y = df["ESBL"].to_numpy(dtype=np.float64); pid = df["patient_id"].to_numpy()
    first = ~df.duplicated("patient_id", keep="first").to_numpy()   # df is sorted by patient, order_dt, order_id
    subsets = {"all": np.ones(len(df), bool), "first_culture_per_patient": first, "episode_index_30d": df["prior_cx_30d"].to_numpy() == 0,
               "screening_agent_tested": df["n_screen_tested"].to_numpy() >= 1}
    if df["esbl_confirm"].notna().any(): subsets["confirmatory_result_available"] = df["esbl_confirm"].notna().to_numpy()
    SENS[tag] = {}
    for name, mask in subsets.items():
        ylab = y.copy()
        if name == "confirmatory_result_available": ylab = df["esbl_confirm"].to_numpy(dtype=np.float64)
        ym, pids = ylab[mask], pid[mask]
        if ym.sum() < 5 or (1 - ym).sum() < 5: continue
        W = boot_weights(pids, key=f"sens_{tgt}_{name}"); SENS[tag][name] = {"n": int(mask.sum()), "n_pos": int(ym.sum()), "prevalence": float(ym.mean())}
        for key in ("FTT", "XGB"):
            p = EXP[tag]["preds"][key]["target_raw"].astype(np.float64)[mask]
            SENS[tag][name][key] = eval_with_ci(ym, p, W, which=("auroc", "auprc"))
        print(f"  {tag} {name:<30s} n={mask.sum():,} ({ym.mean():.1%}) FTT {SENS[tag][name]['FTT']['auroc']:.4f} [{SENS[tag][name]['FTT']['ci_auroc'][0]:.4f}-{SENS[tag][name]['FTT']['ci_auroc'][1]:.4f}] "
              f"XGB {SENS[tag][name]['XGB']['auroc']:.4f}")
rset("sensitivity", value=SENS)
LABELS = {"all": "All cultures", "first_culture_per_patient": "First culture per patient", "episode_index_30d": "Index culture per 30-day episode",
          "screening_agent_tested": r"$\geq$1 CLSI screening agent tested", "confirmatory_result_available": "Confirmatory ESBL result available (label = confirmatory result)"}
lines = []
for s, t in DIRS:
    tag = dir_tag(s, t); lines.append(r"\midrule" + "\n" + r"\multicolumn{5}{l}{\textit{" + f"{s} to {t}" + r"}} \\")
    for name, v in SENS[tag].items():
        lines.append(f"\\quad {LABELS[name]} & {v['n']:,} ({v['prevalence']:.1%}) & {v['FTT']['auroc']:.3f} [{v['FTT']['ci_auroc'][0]:.3f}--{v['FTT']['ci_auroc'][1]:.3f}] & "
                     f"{v['FTT']['auprc']:.3f} & {v['XGB']['auroc']:.3f} [{v['XGB']['ci_auroc'][0]:.3f}--{v['XGB']['ci_auroc'][1]:.3f}] \\\\".replace("%)", r"\%)"))
write_tex("table_sensitivity.tex", "\n".join(lines) + "\n"); save_results(); print(f"Sensitivity analyses done in {time.time()-t0:.0f}s")

  MGB2Stanford all                            n=76,244 (9.2%) FTT 0.7344 [0.7259-0.7428] XGB 0.7266


  MGB2Stanford first_culture_per_patient      n=47,082 (7.5%) FTT 0.6357 [0.6255-0.6453] XGB 0.6138


  MGB2Stanford episode_index_30d              n=70,453 (8.5%) FTT 0.7142 [0.7047-0.7234] XGB 0.7043


  MGB2Stanford screening_agent_tested         n=60,294 (11.6%) FTT 0.7334 [0.7249-0.7415] XGB 0.7344


  Stanford2MGB all                            n=120,742 (13.8%) FTT 0.7516 [0.7454-0.7575] XGB 0.7474


  Stanford2MGB first_culture_per_patient      n=67,185 (10.9%) FTT 0.5977 [0.5904-0.6051] XGB 0.5916


  Stanford2MGB episode_index_30d              n=99,706 (11.1%) FTT 0.6958 [0.6889-0.7028] XGB 0.6942


  Stanford2MGB screening_agent_tested         n=102,742 (16.2%) FTT 0.7464 [0.7403-0.7524] XGB 0.7395


  Stanford2MGB confirmatory_result_available  n=8,166 (6.8%) FTT 0.7659 [0.7331-0.7952] XGB 0.7646
  wrote cross_site_outputs/results/table_sensitivity.tex
Sensitivity analyses done in 62s


In [24]:
# ══════════════════════════════════════════════════════════════════════════════
# Calibration comparison on the target TEST split for all models:
#   raw scores | isotonic fitted on the source calibration split | intercept-only recalibration fitted on the target calibration split
# ══════════════════════════════════════════════════════════════════════════════
t0 = time.time(); CALIB = {}
for src, tgt in DIRS:
    tag = dir_tag(src, tgt); df = SITE_DF[tgt]; y = df["ESBL"].to_numpy(dtype=np.float64); st = df["split"].to_numpy(); te, vc = st == "test", st == "val_cal"
    CALIB[tag] = {"prevalence_test": float(y[te].mean()), "brier_baseline_test": float(y[te].mean() * (1 - y[te].mean()))}
    for key in MODEL_KEYS:
        if key not in EXP[tag]["preds"]: continue
        pr = EXP[tag]["preds"][key]; p_raw = pr["target_raw"].astype(np.float64); p_iso = pr["target_cal"].astype(np.float64)
        recal = fit_intercept_recal(p_raw[vc], y[vc]); p_int = recal(p_raw)
        CALIB[tag][key] = {"auroc": float(roc_auc_score(y[te], p_raw[te])), "raw": calibration_summary(y[te], p_raw[te]),
                           "isotonic_source": calibration_summary(y[te], p_iso[te]), "intercept_target": calibration_summary(y[te], p_int[te])}
        c = CALIB[tag][key]
        print(f"  {tag} {MODEL_NAMES[key]:<16s} Brier raw {c['raw']['brier']:.4f} (ECE {c['raw']['ece']:.3f}, slope {c['raw']['slope']:.2f}) | isotonic {c['isotonic_source']['brier']:.4f} (ECE {c['isotonic_source']['ece']:.3f}) | intercept {c['intercept_target']['brier']:.4f}")
rset("calibration", value=CALIB)
lines = []
for s, t in DIRS:
    tag = dir_tag(s, t); lines.append(r"\midrule" + "\n" + r"\multicolumn{8}{l}{\textit{" + f"{s} to {t} (target test split, prevalence {CALIB[tag]['prevalence_test']:.1%}, prevalence-baseline Brier {CALIB[tag]['brier_baseline_test']:.3f})".replace("%", r"\%") + r"}} \\")
    for key in MODEL_KEYS:
        if key not in CALIB[tag]: continue
        c = CALIB[tag][key]
        lines.append(f"{MODEL_NAMES[key]} & {c['auroc']:.3f} & {c['raw']['brier']:.3f} & {c['raw']['ece']:.3f} & {c['raw']['slope']:.2f} & {c['isotonic_source']['brier']:.3f} & {c['isotonic_source']['ece']:.3f} & {c['intercept_target']['brier']:.3f} \\\\")
write_tex("table_calibration.tex", "\n".join(lines) + "\n"); save_results(); print(f"Calibration table done in {time.time()-t0:.0f}s")

  MGB2Stanford LR               Brier raw 0.1360 (ECE 0.238, slope 1.05) | isotonic 0.0739 (ECE 0.013) | intercept 0.0740
  MGB2Stanford XGBoost          Brier raw 0.1531 (ECE 0.268, slope 1.04) | isotonic 0.0738 (ECE 0.011) | intercept 0.0734
  MGB2Stanford FT-Transformer   Brier raw 0.1383 (ECE 0.249, slope 2.18) | isotonic 0.0736 (ECE 0.012) | intercept 0.0789


  MGB2Stanford MHCA-VAE         Brier raw 0.1321 (ECE 0.235, slope 2.26) | isotonic 0.0750 (ECE 0.012) | intercept 0.0800
  MGB2Stanford DA-VAE           Brier raw 0.3761 (ECE 0.501, slope 0.63) | isotonic 0.0746 (ECE 0.014) | intercept 0.0747
  MGB2Stanford FTT-DANN         Brier raw 0.1340 (ECE 0.240, slope 2.52) | isotonic 0.0760 (ECE 0.018) | intercept 0.0802


  MGB2Stanford TabPFN v2.6      Brier raw 0.0733 (ECE 0.008, slope 0.98) | isotonic 0.0736 (ECE 0.013) | intercept 0.0733
  Stanford2MGB LR               Brier raw 0.1725 (ECE 0.264, slope 0.91) | isotonic 0.0974 (ECE 0.039) | intercept 0.0956


  Stanford2MGB XGBoost          Brier raw 0.1823 (ECE 0.285, slope 0.89) | isotonic 0.0974 (ECE 0.032) | intercept 0.0962
  Stanford2MGB FT-Transformer   Brier raw 0.1523 (ECE 0.228, slope 2.38) | isotonic 0.0961 (ECE 0.028) | intercept 0.1052


  Stanford2MGB MHCA-VAE         Brier raw 0.1581 (ECE 0.239, slope 2.19) | isotonic 0.0979 (ECE 0.024) | intercept 0.1052
  Stanford2MGB DA-VAE           Brier raw 0.6236 (ECE 0.716, slope 0.88) | isotonic 0.0975 (ECE 0.029) | intercept 0.0969


  Stanford2MGB FTT-DANN         Brier raw 0.1524 (ECE 0.224, slope 2.60) | isotonic 0.0988 (ECE 0.037) | intercept 0.1073
  Stanford2MGB TabPFN v2.6      Brier raw 0.0968 (ECE 0.041, slope 0.83) | isotonic 0.0963 (ECE 0.031) | intercept 0.0959
  wrote cross_site_outputs/results/table_calibration.tex
Calibration table done in 1s


In [25]:
# ══════════════════════════════════════════════════════════════════════════════
# Precision-recall diagnostics: what are the highest-scored target cultures, and are they out of the source range?
# ══════════════════════════════════════════════════════════════════════════════
PRDIAG = {}
for src, tgt in DIRS:
    tag = dir_tag(src, tgt); df = SITE_DF[tgt]; y = df["ESBL"].to_numpy(); sc = EXP[tag]["scaler"]
    Xt_ = sc.transform(df[FEATURES].to_numpy(dtype=np.float32)); Xs, ys, ps, ss = site_arrays(SITE_DF[src]); Xs_ = sc.transform(Xs[ss == "train"])
    lo, hi = Xs_.min(axis=0), Xs_.max(axis=0)
    out = {}
    for key in ("FTT", "LR", "XGB", "MHCA-VAE"):
        if key not in EXP[tag]["preds"]: continue
        p = EXP[tag]["preds"][key]["target_raw"]; order = np.argsort(-p, kind="mergesort")
        top20, top50 = order[:20], order[:50]
        oor = ((Xt_[top20] < lo) | (Xt_[top20] > hi)).sum(axis=1)   # features outside the source training range
        extreme = pd.Series(np.abs(Xt_[top20]).mean(axis=0), index=FEATURES).sort_values(ascending=False).head(5)
        out[key] = {"top20_false_positives": int((y[top20] == 0).sum()), "top50_false_positives": int((y[top50] == 0).sum()),
                    "top20_mean_features_outside_source_range": float(oor.mean()), "top20_max_abs_z_features": {k: float(v) for k, v in extreme.items()},
                    "top1_score": float(p[order[0]]), "top1_label": int(y[order[0]])}
        if key == "FTT":
            rows = df.iloc[top20][["order_id", "ESBL", "prior_ESBL", "prior_esbl_ast", "prior_n_resistant_abx", "num_prior_orgs", "days_since_prior_org", "T_ceph3_90d", "T_fq_90d", "elixhauser_count"]].copy()
            rows.insert(1, "score", p[top20]); rows.insert(2, "n_features_out_of_source_range", oor); rows.to_csv(f"{RES_DIR}/pr_diag_{tag}_FTT_top20.csv", index=False)
    PRDIAG[tag] = out
    print(f"{tag}: " + "; ".join(f"{k}: {v['top20_false_positives']}/20 FP in top 20, mean {v['top20_mean_features_outside_source_range']:.1f} features out of source range" for k, v in out.items()))
rset("pr_diagnostics", value=PRDIAG); save_results()

MGB2Stanford: FTT: 4/20 FP in top 20, mean 0.2 features out of source range; LR: 3/20 FP in top 20, mean 0.1 features out of source range; XGB: 3/20 FP in top 20, mean 0.1 features out of source range; MHCA-VAE: 7/20 FP in top 20, mean 0.0 features out of source range


Stanford2MGB: FTT: 2/20 FP in top 20, mean 0.0 features out of source range; LR: 1/20 FP in top 20, mean 0.1 features out of source range; XGB: 1/20 FP in top 20, mean 0.1 features out of source range; MHCA-VAE: 5/20 FP in top 20, mean 0.1 features out of source range


In [26]:
# ══════════════════════════════════════════════════════════════════════════════
# Supplementary Figure 1 (feature importance, both directions) and Supplementary Figure 2 (raw calibration, all models)
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(16, 12)); fig.patch.set_facecolor("white")
for row, (src, tgt) in enumerate(DIRS):
    tag = dir_tag(src, tgt)
    score = EXP[tag]["xgb"].get_booster().get_score(importance_type="weight")
    xi = pd.Series({f"f{i}" if f"f{i}" in score else f: score.get(f"f{i}", score.get(f, 0)) for i, f in enumerate(FEATURES)})
    xi.index = FEATURES; xi = xi.sort_values(ascending=False).head(20)
    ax = axes[row, 0]; ax.barh(xi.index[::-1], xi.values[::-1], color="#e74c3c", alpha=0.85)
    ax.set_xlabel("XGBoost split count (importance_type = weight)"); ax.set_title(f"{'AB'[row]}1. XGBoost top-20, {src}-trained", fontweight="bold")
    top = IMPORTANCE[tag].head(10).sort_values("importance")
    ax = axes[row, 1]; ax.barh(top["feature"], top["importance"], color="#3498db", alpha=0.85)
    ax.set_xlabel(f"FTT permutation importance (AUROC drop, {src} held-out test split)"); ax.set_title(f"{'AB'[row]}2. FT-Transformer top-10, {src} -> {tgt}", fontweight="bold")
    for a in axes[row]: a.spines["top"].set_visible(False); a.spines["right"].set_visible(False)
plt.suptitle("Supplementary Figure 1: feature importance by direction", fontsize=13, fontweight="bold"); plt.tight_layout()
plt.savefig(f"{PAPER_FIG_DIR}/suppfig1_xgb_vs_ftt_importance.png", dpi=300, bbox_inches="tight", facecolor="white"); plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(16, 7)); fig.patch.set_facecolor("white")
for ax, (src, tgt) in zip(axes, DIRS):
    tag = dir_tag(src, tgt); y = SITE_DF[tgt]["ESBL"].to_numpy(dtype=np.float64)
    raw = {k: EXP[tag]["preds"][k]["target_raw"] for k in MODEL_KEYS if k in EXP[tag]["preds"]}
    plot_reliability(ax, y, raw, n_bins=10, min_n=30, annotate=False)
    ax.axhline(y.mean(), color="gray", lw=0.8, ls=":", alpha=0.6)
    ax.set_title(f"{src} -> {tgt}: raw scores (prevalence {y.mean():.1%}, baseline Brier {y.mean()*(1-y.mean()):.3f})", fontweight="bold")
plt.suptitle("Supplementary Figure 2: reliability diagrams of raw model outputs (quantile bins; open markers = bins with < 30 cultures)", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.savefig(f"{PAPER_FIG_DIR}/suppfig2_calibration.png", dpi=300, bbox_inches="tight", facecolor="white"); plt.close(fig)
print("Supplementary Figures 1 and 2 saved")

Supplementary Figures 1 and 2 saved


In [27]:
# ══════════════════════════════════════════════════════════════════════════════
# Finalise: results.json, manifest, figure check against the manuscript sources
# ══════════════════════════════════════════════════════════════════════════════
save_results()
figs = sorted(glob.glob(f"{PAPER_FIG_DIR}/*.png")); tables = sorted(glob.glob(f"{RES_DIR}/*.tex"))
print("Figures:"); [print(f"  {f}  {os.path.getsize(f)/1e6:.2f} MB  {time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(f)))}") for f in figs]
print("Tables:"); [print(f"  {t}") for t in tables]
needed = set()
for tex in glob.glob("figures/*.tex"):
    needed |= set(re.findall(r"\\includegraphics\[[^\]]*\]\{([^}]+)\}", open(tex).read()))
missing = [n for n in needed if not os.path.exists(f"figures/{n}")]
print(f"Manuscript figure references: {sorted(needed)}")
print("MISSING figure files referenced by figures/*.tex: " + (", ".join(sorted(missing)) if missing else "none"))
rset("manifest", value={"figures": figs, "tables": tables, "missing_manuscript_figures": missing, "finished": time.strftime("%Y-%m-%d %H:%M:%S")})
save_results()
print(f"\nRESULTS written to {RESULTS_PATH}. Done.")

Figures:
  figures/fig1a_mgb_to_stanford.png  0.99 MB  2026-09-16 05:10
  figures/fig1b_stanford_to_mgb.png  1.04 MB  2026-09-16 05:11
  figures/ftt_importance_m2s.png  0.19 MB  2026-09-16 05:20
  figures/ftt_importance_s2m.png  0.19 MB  2026-09-16 05:20
  figures/multimodel_transfer.png  0.96 MB  2026-09-16 05:20
  figures/suppfig1_xgb_vs_ftt_importance.png  0.69 MB  2026-09-16 05:34
  figures/suppfig2_calibration.png  0.59 MB  2026-09-16 05:34
  figures/suppfig3_subgroups.png  0.34 MB  2026-09-16 05:33
  figures/suppfig4_reduced_features.png  0.64 MB  2026-09-16 05:32
Tables:
  cross_site_outputs/results/table1.tex
  cross_site_outputs/results/table2.tex
  cross_site_outputs/results/table3_delta.tex
  cross_site_outputs/results/table_calibration.tex
  cross_site_outputs/results/table_clinical.tex
  cross_site_outputs/results/table_reduced.tex
  cross_site_outputs/results/table_sensitivity.tex
  cross_site_outputs/results/table_subgroup.tex
  cross_site_outputs/results/table_transfer.